# Transformer Bushing FEA: Static, Modal and Thermal-Structural

Simplified dry-type bushing (copper conductor, epoxy core, aluminium flange) analysed with open-source tools:
Gmsh for geometry and mesh, FEniCSx (dolfinx 0.11) for the finite element analysis, SLEPc for the modal analysis,
PyVista and Matplotlib for figures. Every number in the generated report and documents comes from
`results/results.json`.

**How to use in Google Colab**
1. Run the first code cell. It installs a conda base (condacolab) and **restarts the runtime once**. This is expected.
2. After the restart, run all cells from the top again (*Runtime > Run all*). The install cell then installs
   dolfinx and the other packages from conda-forge (a few minutes) and is skipped on later runs.
3. The full study takes roughly 15 to 30 minutes. The fine mesh of the convergence study is chosen from the
   available RAM (see the run cell).

The load cases are based on the type of checks in IEC 60137 and IEEE 693; the study does not claim compliance,
qualification or a rated design. All material values are typical literature values, assumed.

## 1. Install (condacolab, then conda-forge packages)

In [ ]:
import importlib.util, sys
if importlib.util.find_spec("dolfinx") is None:
    if "google.colab" in sys.modules:
        if importlib.util.find_spec("condacolab") is None:
            !pip install -q condacolab
        import condacolab
        if not condacolab.check():
            condacolab.install()          # restarts the runtime; run all cells again afterwards
    else:
        raise RuntimeError("dolfinx not found: create the conda environment from environment.yml first")
else:
    import dolfinx
    print("dolfinx", dolfinx.__version__, "already installed")

In [ ]:
import importlib.util
if importlib.util.find_spec("dolfinx") is None:
    !mamba install -y -q -c conda-forge "fenics-dolfinx=0.11" mpich petsc4py slepc4py gmsh python-gmsh pyvista matplotlib numpy scipy pyyaml pandas
import dolfinx, gmsh
from petsc4py import PETSc
print("dolfinx", dolfinx.__version__, "| gmsh", gmsh.__version__, "| PETSc", PETSc.Sys.getVersion())

## 2. Project folder

In [ ]:
import os
os.makedirs("bushing-fea/src", exist_ok=True)
os.makedirs("bushing-fea/docs/templates", exist_ok=True)
os.makedirs("bushing-fea/report", exist_ok=True)
os.makedirs("bushing-fea/tools", exist_ok=True)
%cd bushing-fea

## 3. Parameters (`config.yaml`) and environment file

In [ ]:
%%writefile config.yaml
# Transformer bushing FEA: all parameters in SI units.
# The bushing axis is the z axis. z = 0 is the bottom face of the insulating core.

geometry:
  conductor:            # central copper tube
    outer_diameter: 0.040      # m
    inner_diameter: 0.030      # m
    length: 1.600              # m, centred on the core, so it spans z = -0.2 to 1.4 m
  core:                 # resin-impregnated epoxy core, solid around the conductor
    outer_diameter: 0.160      # m
    length: 1.200              # m, spans z = 0 to 1.2 m
  flange:               # aluminium mounting ring bonded to the core
    outer_diameter: 0.300      # m (inner diameter equals the core outer diameter)
    thickness: 0.030           # m
    z_bottom: 0.400            # m, bottom face of the flange measured from the core bottom

mesh:
  h: 0.020                     # m, global element size: the single mesh-size parameter
  conductor_size_ratio: 0.30   # element size at the conductor surface = ratio * h
  flange_size_ratio: 0.50      # element size in the flange zone = ratio * h
  refine_distance_ratio: 3.0   # size grows back to h over this many h from the conductor
  element_order: 2             # quadratic (10-node) tetrahedra, quadratic displacement/temperature
  convergence_h: [0.030, 0.020, 0.0135]   # m, coarse, medium, fine for the convergence study

materials:
  # Every value below is a typical literature value, assumed. Not taken from a datasheet.
  copper:
    E: 117.0e+9          # Pa, typical literature value, assumed
    nu: 0.34            # -, typical literature value, assumed
    rho: 8960.0         # kg/m3, typical literature value, assumed
    k: 390.0            # W/(m K), typical literature value, assumed
    alpha: 17.0e-6      # 1/K, typical literature value, assumed
    resistivity: 1.72e-8  # Ohm m at 20 C, typical literature value, assumed
  epoxy:
    E: 12.0e+9           # Pa, typical literature value, assumed
    nu: 0.35            # -, typical literature value, assumed
    rho: 1800.0         # kg/m3, typical literature value, assumed
    k: 0.6              # W/(m K), typical literature value, assumed
    alpha: 35.0e-6      # 1/K, typical literature value, assumed
  aluminium:
    E: 70.0e+9           # Pa, typical literature value, assumed
    nu: 0.33            # -, typical literature value, assumed
    rho: 2700.0         # kg/m3, typical literature value, assumed
    k: 200.0            # W/(m K), typical literature value, assumed
    alpha: 23.0e-6      # 1/K, typical literature value, assumed

loads:
  g: 9.81               # m/s2
  cantilever_force: 1000.0   # N, horizontal (x direction) on the top conductor end face.
                             # Illustrative value; the IEC 60137 test load depends on rating
                             # and should be checked in the standard.
  seismic_horizontal_g: 0.5  # horizontal (x direction) acceleration in g for load case S3.
                             # Simplified static-equivalent check, not a seismic qualification.
  current: 1250.0       # A, load current for Joule heating
  h_conv: 10.0          # W/(m2 K), natural convection coefficient on core and flange outer surfaces
  T_ambient: 40.0       # C, ambient air temperature
  T_reference: 20.0     # C, stress-free reference temperature for thermal strain

modal:
  n_modes: 6

postprocessing:
  # Stress at re-entrant corners is singular and does not converge. A second "away from
  # corners" maximum excludes points closer than this distance to the singular ring edges.
  corner_exclusion: 0.010    # m
  radial_line_points: 200
  axial_line_points: 400

In [ ]:
%%writefile environment.yml
name: bushing-fea
channels:
  - conda-forge
dependencies:
  - python=3.12
  - fenics-dolfinx=0.11   # code written for the 0.11 API
  - mpich
  - petsc4py
  - slepc4py
  - gmsh
  - python-gmsh
  - pyvista
  - matplotlib
  - numpy
  - scipy
  - pyyaml
  - pandas

## 4. Source code
`common.py` holds shared helpers (tags, material fields, stress, point evaluation, results file).

In [ ]:
%%writefile src/common.py
"""Shared helpers: configuration, tags, material fields, stress, point evaluation, results file."""
import json
from pathlib import Path

import numpy as np
import ufl
import yaml
from dolfinx import fem, geometry
from mpi4py import MPI

ROOT = Path(__file__).resolve().parent.parent
RESULTS_DIR = ROOT / "results"
FIGURES_DIR = ROOT / "figures"
RESULTS_FILE = RESULTS_DIR / "results.json"

# Physical group tags (shared by geometry.py and the solvers)
TAGS = {
    "conductor": 1, "core": 2, "flange": 3,                    # volumes
    "fixed": 11,               # flange bottom face (fixed support; also convective in the thermal case)
    "top_end": 12,             # top conductor end face (terminal load)
    "convection": 13,          # outer surfaces of core and flange, except the flange bottom face
    "bore": 14,                # conductor bore (insulated)
    "conductor_exposed": 15,   # conductor outer surface outside the core and the bottom end face
    "if_conductor_core": 21,   # conductor to core interface
    "if_core_flange": 22,      # core to flange interface
}
BODIES = {"conductor": "copper", "core": "epoxy", "flange": "aluminium"}


def load_config():
    with open(ROOT / "config.yaml") as f:
        cfg = yaml.safe_load(f)
    for mat, props in cfg["materials"].items():
        for k, v in props.items():
            if not isinstance(v, (int, float)):
                raise TypeError(f"materials.{mat}.{k} is not a number: {v!r} (write exponents as e+9)")
    return cfg


def derived_geometry(cfg):
    """Radii and axial positions derived from config.yaml (z = 0 at the core bottom)."""
    g = cfg["geometry"]
    d = {
        "r_ci": g["conductor"]["inner_diameter"] / 2,
        "r_co": g["conductor"]["outer_diameter"] / 2,
        "r_k": g["core"]["outer_diameter"] / 2,
        "r_f": g["flange"]["outer_diameter"] / 2,
        "L_core": g["core"]["length"],
        "L_cond": g["conductor"]["length"],
        "z_f0": g["flange"]["z_bottom"],
        "t_f": g["flange"]["thickness"],
    }
    d["z_c0"] = d["L_core"] / 2 - d["L_cond"] / 2       # conductor bottom
    d["z_c1"] = d["L_core"] / 2 + d["L_cond"] / 2       # conductor top
    d["z_f1"] = d["z_f0"] + d["t_f"]
    return d


def singular_rings(cfg):
    """(r, z) of ring edges where the elastic stress field is singular (re-entrant or
    bi-material corners and the edges of the fixed face)."""
    d = derived_geometry(cfg)
    return [
        (d["r_k"], d["z_f0"]),   # flange bottom inner edge / fixed face inner edge
        (d["r_k"], d["z_f1"]),   # flange top inner edge (re-entrant)
        (d["r_f"], d["z_f0"]),   # fixed face outer edge
        (d["r_co"], d["L_core"]),  # conductor exits the core top
        (d["r_co"], 0.0),        # conductor exits the core bottom
    ]


def material_field(msh, cell_tags, cfg, prop):
    """Piecewise-constant (DG0) field of a material property, assigned by body."""
    Q = fem.functionspace(msh, ("DG", 0))
    f = fem.Function(Q)
    for body, mat in BODIES.items():
        cells = cell_tags.find(TAGS[body])
        f.x.array[cells] = cfg["materials"][mat][prop]
    f.x.scatter_forward()
    return f


def lame(E, nu):
    return E * nu / ((1 + nu) * (1 - 2 * nu)), E / (2 * (1 + nu))


def eps(u):
    return ufl.sym(ufl.grad(u))


def sigma(u, E, nu, dT=None, alpha=None):
    lmbda, mu = lame(E, nu)
    e = eps(u)
    if dT is not None:
        e = e - alpha * dT * ufl.Identity(3)
    return 2 * mu * e + lmbda * ufl.tr(e) * ufl.Identity(3)


def von_mises(s):
    dev = s - ufl.tr(s) / 3 * ufl.Identity(3)
    return ufl.sqrt(1.5 * ufl.inner(dev, dev))


def project_dg1(expr, msh):
    """Interpolate a scalar UFL expression into discontinuous P1 (per-body values are kept)."""
    W = fem.functionspace(msh, ("DG", 1))
    f = fem.Function(W)
    f.interpolate(fem.Expression(expr, W.element.interpolation_points))
    return f


def body_max(f, cell_tags, body, exclude_rings=None, exclusion=0.0):
    """Maximum of a DG field in one body and its location; optionally skip points close
    to singular ring edges."""
    W = f.function_space
    cells = cell_tags.find(TAGS[body])
    dofs = np.unique(W.dofmap.list[cells].flatten())
    X = W.tabulate_dof_coordinates()[dofs]
    vals = f.x.array[dofs]
    if exclude_rings:
        r = np.hypot(X[:, 0], X[:, 1])
        keep = np.ones(len(dofs), dtype=bool)
        for (r0, z0) in exclude_rings:
            keep &= np.hypot(r - r0, X[:, 2] - z0) > exclusion
        X, vals = X[keep], vals[keep]
    i = int(np.argmax(vals))
    return float(vals[i]), location(X[i])


def location(x):
    x = np.asarray(x, dtype=float)
    return {"x": round(float(x[0]), 4), "y": round(float(x[1]), 4), "z": round(float(x[2]), 4),
            "r": round(float(np.hypot(x[0], x[1])), 4)}


def eval_points(u, points):
    """Evaluate a Function at points (N, 3); NaN where a point is outside the mesh."""
    msh = u.function_space.mesh
    points = np.asarray(points, dtype=float)
    tree = geometry.bb_tree(msh, msh.topology.dim)
    cand = geometry.compute_collisions_points(tree, points)
    coll = geometry.compute_colliding_cells(msh, cand, points)
    out = np.full((len(points), int(np.prod(u.ufl_shape)) or 1), np.nan)
    idx, cells = [], []
    for i in range(len(points)):
        c = coll.links(i)
        if len(c) > 0:
            idx.append(i)
            cells.append(c[0])
    if idx:
        out[idx] = u.eval(points[idx], np.array(cells, dtype=np.int32))
    return out


def surface_average(expr, ds_tag, msh):
    num = fem.assemble_scalar(fem.form(expr * ds_tag))
    den = fem.assemble_scalar(fem.form(fem.Constant(msh, 1.0) * ds_tag))
    return float(msh.comm.allreduce(num, op=MPI.SUM) / msh.comm.allreduce(den, op=MPI.SUM))


def nodal_max_displacement(u):
    """Largest displacement magnitude at a node of the displacement field and its location."""
    V = u.function_space
    X = V.tabulate_dof_coordinates()
    a = u.x.array.reshape(-1, 3)
    n = a.shape[0]
    mag = np.linalg.norm(a, axis=1)
    i = int(np.argmax(mag))
    return float(mag[i]), location(X[:n][i])


def update_results(section, data):
    """Merge one section into results/results.json."""
    RESULTS_DIR.mkdir(exist_ok=True)
    res = json.loads(RESULTS_FILE.read_text()) if RESULTS_FILE.exists() else {}
    res[section] = data
    RESULTS_FILE.write_text(json.dumps(res, indent=2))


def read_results():
    return json.loads(RESULTS_FILE.read_text())


def pct_change(new, ref):
    return 100.0 * (new - ref) / ref

### `smoke_test.py`: Stage 0 smoke test: Poisson problem on the unit cube

In [ ]:
%%writefile src/smoke_test.py
"""Smoke test: Poisson problem on the unit cube with a known exact solution."""
import numpy as np
import dolfinx
import ufl
from mpi4py import MPI
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

print("dolfinx version:", dolfinx.__version__)
msh = mesh.create_unit_cube(MPI.COMM_WORLD, 8, 8, 8)
V = fem.functionspace(msh, ("Lagrange", 2))
x = ufl.SpatialCoordinate(msh)
u_exact = 1 + x[0] ** 2 + 2 * x[1] ** 2 + 3 * x[2] ** 2
f = fem.Constant(msh, -12.0)
u_bc = fem.Function(V)
u_bc.interpolate(lambda X: 1 + X[0] ** 2 + 2 * X[1] ** 2 + 3 * X[2] ** 2)
msh.topology.create_connectivity(2, 3)
facets = mesh.exterior_facet_indices(msh.topology)
bc = fem.dirichletbc(u_bc, fem.locate_dofs_topological(V, 2, facets))
u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
problem = LinearProblem(ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx, f * v * ufl.dx, bcs=[bc],
                        petsc_options={"ksp_type": "preonly", "pc_type": "lu"}, petsc_options_prefix="smoke_")
uh = problem.solve()
err = np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form((uh - u_exact) ** 2 * ufl.dx)), op=MPI.SUM))
print(f"L2 error vs exact quadratic solution: {err:.3e}")
assert err < 1e-8
print("Smoke test passed")

### `geometry.py`: Stage 2: geometry, physical groups and quadratic tetrahedral mesh (Gmsh)

In [ ]:
%%writefile src/geometry.py
"""Stage 2: simplified dry-type condenser bushing in Gmsh (OpenCASCADE kernel).

Three coaxial bodies along z: copper conductor tube, epoxy core, aluminium flange.
The bodies are fragmented so the mesh is conforming at the bonded interfaces.
"""
import sys

import gmsh
import numpy as np
from mpi4py import MPI

from common import RESULTS_DIR, TAGS, derived_geometry, load_config, update_results

TOL = 1e-6


def _classify_surface(tag, d):
    """Return the physical group name of an exterior or interface surface from its bounding box."""
    x0, y0, z0, x1, y1, z1 = gmsh.model.getBoundingBox(2, tag)
    r = max(abs(x0), abs(x1), abs(y0), abs(y1))
    flat = abs(z1 - z0) < TOL

    def near(a, b):
        return abs(a - b) < 1e-4

    if flat:
        z = z0
        if near(z, d["z_f0"]) and near(r, d["r_f"]):
            return "fixed"
        if near(z, d["z_c1"]):
            return "top_end"
        if near(z, d["z_c0"]):
            return "conductor_exposed"
        return "convection"          # core bottom/top annulus, flange top annulus
    if near(r, d["r_ci"]):
        return "bore"
    if near(r, d["r_co"]):
        if z0 > -TOL and z1 < d["L_core"] + TOL:
            return "if_conductor_core"
        return "conductor_exposed"
    if near(r, d["r_k"]):
        if z0 > d["z_f0"] - TOL and z1 < d["z_f1"] + TOL:
            return "if_core_flange"
        return "convection"
    if near(r, d["r_f"]):
        return "convection"
    raise RuntimeError(f"Unclassified surface {tag}: bbox {x0, y0, z0, x1, y1, z1}")


def build_model(cfg, h=None):
    """Create geometry, physical groups and a quadratic tetrahedral mesh in the active gmsh model.

    Returns a dictionary with mesh statistics.
    """
    d = derived_geometry(cfg)
    mc = cfg["mesh"]
    h = mc["h"] if h is None else h
    occ = gmsh.model.occ

    # Conductor tube
    c_out = occ.addCylinder(0, 0, d["z_c0"], 0, 0, d["L_cond"], d["r_co"])
    c_in = occ.addCylinder(0, 0, d["z_c0"], 0, 0, d["L_cond"], d["r_ci"])
    cond, _ = occ.cut([(3, c_out)], [(3, c_in)])
    # Epoxy core: solid around the conductor (annulus from conductor outer radius)
    k_out = occ.addCylinder(0, 0, 0, 0, 0, d["L_core"], d["r_k"])
    k_in = occ.addCylinder(0, 0, 0, 0, 0, d["L_core"], d["r_co"])
    core, _ = occ.cut([(3, k_out)], [(3, k_in)])
    # Flange ring
    f_out = occ.addCylinder(0, 0, d["z_f0"], 0, 0, d["t_f"], d["r_f"])
    f_in = occ.addCylinder(0, 0, d["z_f0"], 0, 0, d["t_f"], d["r_k"])
    flange, _ = occ.cut([(3, f_out)], [(3, f_in)])
    occ.fragment(cond + core + flange, [])
    occ.synchronize()

    # Identify volumes by their centre of mass radius
    vols = {}
    for _, v in gmsh.model.getEntities(3):
        x, y, z = gmsh.model.occ.getCenterOfMass(3, v)
        x0, y0, z0, x1, y1, z1 = gmsh.model.getBoundingBox(3, v)
        rmax = max(abs(x0), abs(x1))
        name = ("conductor" if abs(rmax - d["r_co"]) < 1e-4 else
                "core" if abs(rmax - d["r_k"]) < 1e-4 else "flange")
        vols.setdefault(name, []).append(v)
    assert sorted(vols) == ["conductor", "core", "flange"], vols
    for name, vs in vols.items():
        gmsh.model.addPhysicalGroup(3, vs, TAGS[name], name)

    surfs = {}
    for _, s in gmsh.model.getEntities(2):
        surfs.setdefault(_classify_surface(s, d), []).append(s)
    for name, ss in surfs.items():
        gmsh.model.addPhysicalGroup(2, ss, TAGS[name], name)

    # Mesh size: global h, refined at the conductor surface and in the flange zone
    cond_surfs = surfs["if_conductor_core"] + [s for s in surfs["conductor_exposed"]
                                               if abs(gmsh.model.getBoundingBox(2, s)[2] -
                                                      gmsh.model.getBoundingBox(2, s)[5]) > TOL]
    fd = gmsh.model.mesh.field
    dist = fd.add("Distance")
    fd.setNumbers(dist, "SurfacesList", cond_surfs)
    fd.setNumber(dist, "Sampling", 200)
    thr = fd.add("Threshold")
    fd.setNumber(thr, "InField", dist)
    fd.setNumber(thr, "SizeMin", mc["conductor_size_ratio"] * h)
    fd.setNumber(thr, "SizeMax", h)
    fd.setNumber(thr, "DistMin", 0.0)
    fd.setNumber(thr, "DistMax", mc["refine_distance_ratio"] * h)
    box = fd.add("Box")
    hf = mc["flange_size_ratio"] * h
    fd.setNumber(box, "VIn", hf)
    fd.setNumber(box, "VOut", h)
    for key, val in [("XMin", -d["r_f"]), ("XMax", d["r_f"]), ("YMin", -d["r_f"]), ("YMax", d["r_f"]),
                     ("ZMin", d["z_f0"] - 2 * hf), ("ZMax", d["z_f1"] + 2 * hf)]:
        fd.setNumber(box, key, val)
    fd.setNumber(box, "Thickness", 2 * h)
    fmin = fd.add("Min")
    fd.setNumbers(fmin, "FieldsList", [thr, box])
    fd.setAsBackgroundMesh(fmin)
    gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)
    gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)
    gmsh.option.setNumber("Mesh.Algorithm3D", 10)  # HXT
    gmsh.option.setNumber("Mesh.Optimize", 1)

    gmsh.model.mesh.generate(3)
    gmsh.model.mesh.setOrder(mc["element_order"])
    gmsh.model.mesh.optimize("HighOrder")  # untangle curved elements if needed

    # Statistics
    node_tags, _, _ = gmsh.model.mesh.getNodes()
    etypes, etags, _ = gmsh.model.mesh.getElements(3)
    tets = np.concatenate(etags)
    q = gmsh.model.mesh.getElementQualities(tets, "minSICN")
    gamma = gmsh.model.mesh.getElementQualities(tets, "gamma")
    stats = {
        "h_global_m": h,
        "h_conductor_m": mc["conductor_size_ratio"] * h,
        "h_flange_m": hf,
        "element_type": gmsh.model.mesh.getElementProperties(int(etypes[0]))[0],
        "n_nodes": int(len(node_tags)),
        "n_elements": int(len(tets)),
        "quality_minSICN_min": float(q.min()),
        "quality_minSICN_mean": float(q.mean()),
        "quality_gamma_min": float(gamma.min()),
        "quality_gamma_mean": float(gamma.mean()),
        "share_minSICN_below_0p2": float(np.mean(q < 0.2)),
    }
    return stats


def create_mesh(cfg, h=None, comm=MPI.COMM_WORLD):
    """Build the model and return (dolfinx MeshData, stats)."""
    from dolfinx.io import gmsh as dgmsh

    gmsh.initialize()
    gmsh.option.setNumber("General.Terminal", 0)
    gmsh.model.add("bushing")
    stats = build_model(cfg, h)
    data = dgmsh.model_to_mesh(gmsh.model, comm, 0, gdim=3)
    gmsh.finalize()
    return data, stats


if __name__ == "__main__":
    cfg = load_config()
    gmsh.initialize(sys.argv)
    gmsh.option.setNumber("General.Terminal", 0)
    gmsh.model.add("bushing")
    stats = build_model(cfg)
    RESULTS_DIR.mkdir(exist_ok=True)
    gmsh.write(str(RESULTS_DIR / "bushing.msh"))
    gmsh.finalize()
    d = derived_geometry(cfg)
    update_results("mesh", stats)
    update_results("geometry", {k: float(v) for k, v in d.items()})
    for k, v in stats.items():
        print(f"{k:28s} {v}")

### `hand_calcs.py`: Analytical checks: beam theory, 1D radial conduction, verification table

In [ ]:
%%writefile src/hand_calcs.py
"""Analytical checks: beam theory for the static and modal cases, 1D radial heat conduction for the
thermal case, free thermal elongation for the thermal-structural case.

All beam checks treat the part above the flange as a cantilever clamped at the flange bottom face
(z = z_f0, the fixed support plane) and loaded at the conductor top (z = z_c1).
"""
import numpy as np
from scipy.linalg import eigh

from common import derived_geometry, load_config, read_results, update_results


def sections(cfg):
    d = derived_geometry(cfg)
    m = cfg["materials"]
    A_c = np.pi * (d["r_co"] ** 2 - d["r_ci"] ** 2)
    A_k = np.pi * (d["r_k"] ** 2 - d["r_co"] ** 2)
    I_c = np.pi / 4 * (d["r_co"] ** 4 - d["r_ci"] ** 4)
    I_k = np.pi / 4 * (d["r_k"] ** 4 - d["r_co"] ** 4)
    EI_c = m["copper"]["E"] * I_c
    EI_k = m["epoxy"]["E"] * I_k
    mu_c = m["copper"]["rho"] * A_c
    mu_k = m["epoxy"]["rho"] * A_k
    return dict(A_c=A_c, A_k=A_k, I_c=I_c, I_k=I_k, EI_c=EI_c, EI_k=EI_k, EI_eff=EI_c + EI_k,
                m_c=mu_c, m_k=mu_k, m_eff=mu_c + mu_k)


def cantilever(cfg):
    d = derived_geometry(cfg)
    s = sections(cfg)
    F = cfg["loads"]["cantilever_force"]
    L = d["z_c1"] - d["z_f0"]            # flange (fixed plane) to conductor top
    a = d["L_core"] - d["z_f0"]          # segment 1: core plus conductor
    b = d["z_c1"] - d["L_core"]          # segment 2: bare conductor
    delta_uniform = F * L ** 3 / (3 * s["EI_eff"])
    EI1, EI2 = s["EI_eff"], s["EI_c"]
    # segment 1 carries shear F and end moment F*b; segment 2 is a cantilever on the rotated end of segment 1
    d1 = F * a ** 3 / (3 * EI1) + F * b * a ** 2 / (2 * EI1)
    th1 = F * a ** 2 / (2 * EI1) + F * b * a / EI1
    d2 = F * b ** 3 / (3 * EI2)
    delta_two = d1 + th1 * b + d2
    # bending stress in the bare conductor at the core top (moment F*b), for reference
    sigma_cond = F * b * d["r_co"] / s["I_c"]
    return {"F_N": F, "L_m": L, "a_core_segment_m": a, "b_bare_conductor_m": b,
            "EI_conductor_Nm2": s["EI_c"], "EI_core_Nm2": s["EI_k"], "EI_eff_Nm2": s["EI_eff"],
            "delta_uniform_m": delta_uniform, "delta_two_segment_m": delta_two,
            "two_segment_parts_m": {"segment1_deflection": d1, "segment1_rotation_times_b": th1 * b,
                                    "segment2_cantilever": d2},
            "bending_stress_conductor_at_core_top_Pa": sigma_cond}


def cantilever_profile(cfg, s_pos):
    """Two-segment beam deflection w(s) under tip load F, s measured from the fixed plane."""
    c = cantilever(cfg)
    F, a, b = c["F_N"], c["a_core_segment_m"], c["b_bare_conductor_m"]
    EI1, EI2 = c["EI_eff_Nm2"], c["EI_conductor_Nm2"]
    L = a + b
    s_pos = np.asarray(s_pos, dtype=float)
    # segment 1: M(s) = F (L - s); w'' = M / EI1
    w1 = F / EI1 * (L * s_pos ** 2 / 2 - s_pos ** 3 / 6)
    wa = F / EI1 * (L * a ** 2 / 2 - a ** 3 / 6)
    ta = F / EI1 * (L * a - a ** 2 / 2)
    t = s_pos - a
    w2 = wa + ta * t + F / EI2 * (b * t ** 2 / 2 - t ** 3 / 6)
    return np.where(s_pos <= a, w1, w2)


N_BEAM_ELEMENTS = 200


def stepped_beam_frequencies(cfg, n_el=N_BEAM_ELEMENTS, n=3):
    """Euler-Bernoulli finite element model (Hermite elements) of the two-segment cantilever."""
    d = derived_geometry(cfg)
    s = sections(cfg)
    a = d["L_core"] - d["z_f0"]
    L = d["z_c1"] - d["z_f0"]
    z = np.linspace(0, L, n_el + 1)
    ndof = 2 * (n_el + 1)
    K = np.zeros((ndof, ndof))
    M = np.zeros((ndof, ndof))
    for e in range(n_el):
        le = z[e + 1] - z[e]
        zm = 0.5 * (z[e] + z[e + 1])
        EI, mu = (s["EI_eff"], s["m_eff"]) if zm < a else (s["EI_c"], s["m_c"])
        ke = EI / le ** 3 * np.array([[12, 6 * le, -12, 6 * le], [6 * le, 4 * le ** 2, -6 * le, 2 * le ** 2],
                                      [-12, -6 * le, 12, -6 * le], [6 * le, 2 * le ** 2, -6 * le, 4 * le ** 2]])
        me = mu * le / 420 * np.array([[156, 22 * le, 54, -13 * le], [22 * le, 4 * le ** 2, 13 * le, -3 * le ** 2],
                                       [54, 13 * le, 156, -22 * le], [-13 * le, -3 * le ** 2, -22 * le, 4 * le ** 2]])
        idx = slice(2 * e, 2 * e + 4)
        K[idx, idx] += ke
        M[idx, idx] += me
    free = slice(2, ndof)  # clamped at z = 0
    w2 = eigh(K[free, free], M[free, free], eigvals_only=True, subset_by_index=[0, n - 1])
    return np.sqrt(w2) / (2 * np.pi)


def modal(cfg):
    d = derived_geometry(cfg)
    s = sections(cfg)
    L = d["z_c1"] - d["z_f0"]
    f1_uniform = 1.875 ** 2 / (2 * np.pi) * np.sqrt(s["EI_eff"] / (s["m_eff"] * L ** 4))
    f_step = stepped_beam_frequencies(cfg)
    return {"L_m": L, "EI_eff_Nm2": s["EI_eff"], "m_eff_kg_per_m": s["m_eff"],
            "f1_uniform_Hz": float(f1_uniform), "f1_two_segment_beam_Hz": float(f_step[0]),
            "f2_bending_two_segment_beam_Hz": float(f_step[1]), "n_beam_elements": N_BEAM_ELEMENTS}


def joule_heating(cfg):
    s = sections(cfg)
    I = cfg["loads"]["current"]
    rho_e = cfg["materials"]["copper"]["resistivity"]
    q = I ** 2 * rho_e / s["A_c"] ** 2
    return {"q_W_per_m3": q, "Q_per_length_W_per_m": q * s["A_c"],
            "Q_total_W": q * s["A_c"] * cfg["geometry"]["conductor"]["length"]}


def radial_temperature(cfg, r, q_scale=1.0):
    """Long cylinder: uniform generation q in the copper tube (insulated bore), conduction through
    the epoxy, convection h to T_amb at the core surface. Returns T(r) in C."""
    d = derived_geometry(cfg)
    m = cfg["materials"]
    ld = cfg["loads"]
    q = joule_heating(cfg)["q_W_per_m3"] * q_scale
    kc, ke, h = m["copper"]["k"], m["epoxy"]["k"], ld["h_conv"]
    rci, rco, rk = d["r_ci"], d["r_co"], d["r_k"]
    Qp = q * np.pi * (rco ** 2 - rci ** 2)
    T_s = ld["T_ambient"] + Qp / (2 * np.pi * rk * h)
    T_co = T_s + Qp / (2 * np.pi * ke) * np.log(rk / rco)
    r = np.asarray(r, dtype=float)
    T = np.where(r >= rco, T_s + Qp / (2 * np.pi * ke) * np.log(rk / np.maximum(r, rco)),
                 T_co + q / (4 * kc) * (rco ** 2 - r ** 2) - q * rci ** 2 / (2 * kc) * np.log(rco / np.maximum(r, rci)))
    return T


def thermal(cfg):
    d = derived_geometry(cfg)
    T = radial_temperature(cfg, [d["r_ci"], d["r_co"], d["r_k"]])
    jh = joule_heating(cfg)
    return {**jh, "T_bore_C": float(T[0]), "T_conductor_surface_C": float(T[1]),
            "T_core_surface_C": float(T[2]),
            "dT_epoxy_K": float(T[1] - T[2]), "dT_convection_K": float(T[2] - cfg["loads"]["T_ambient"])}


def run(cfg):
    out = {"sections": sections(cfg), "cantilever": cantilever(cfg), "modal": modal(cfg), "thermal": thermal(cfg)}
    update_results("hand_calcs", out)
    return out


def pct(fea, ref):
    return 100.0 * (fea - ref) / ref


def verify():
    """Compare hand calculations with FEA results stored in results.json; writes a 'verification' table."""
    r = read_results()
    hc = r["hand_calcs"]
    rows = []
    fea = r["static"]["S1_cantilever"]["top_deflection_x_m"]
    rows.append({"check": "S1 top deflection, uniform composite cantilever", "quantity": "deflection",
                 "unit": "mm", "analytical": hc["cantilever"]["delta_uniform_m"] * 1e3, "fea": fea * 1e3})
    rows.append({"check": "S1 top deflection, two-segment cantilever", "quantity": "deflection",
                 "unit": "mm", "analytical": hc["cantilever"]["delta_two_segment_m"] * 1e3, "fea": fea * 1e3})
    f1 = r["modal"]["modes"][0]["frequency_Hz"]
    rows.append({"check": "f1, uniform composite cantilever", "quantity": "frequency", "unit": "Hz",
                 "analytical": hc["modal"]["f1_uniform_Hz"], "fea": f1})
    rows.append({"check": "f1, two-segment beam (1D beam elements)", "quantity": "frequency", "unit": "Hz",
                 "analytical": hc["modal"]["f1_two_segment_beam_Hz"], "fea": f1})
    th = r["thermal"]
    Ta = r["inputs"]["loads"]["T_ambient"] if "inputs" in r else load_config()["loads"]["T_ambient"]
    rows.append({"check": "Conductor surface temperature rise above ambient at core mid-height, 1D radial",
                 "quantity": "temperature rise", "unit": "K",
                 "analytical": hc["thermal"]["T_conductor_surface_C"] - Ta, "fea": th["mid_height"]["T_conductor_surface_C"] - Ta})
    rows.append({"check": "Core surface temperature rise above ambient at core mid-height, 1D radial",
                 "quantity": "temperature rise", "unit": "K",
                 "analytical": hc["thermal"]["T_core_surface_C"] - Ta, "fea": th["mid_height"]["T_core_surface_C"] - Ta})
    rows.append({"check": "Temperature rise across epoxy at mid-height, 1D radial", "quantity": "temperature difference",
                 "unit": "K", "analytical": hc["thermal"]["dT_epoxy_K"], "fea": th["mid_height"]["dT_epoxy_K"]})
    rows.append({"check": "Temperature rise across epoxy at mid-height, 1D radial with FEA local heat flow",
                 "quantity": "temperature difference", "unit": "K",
                 "analytical": th["mid_height"]["dT_epoxy_1D_with_local_heat_flow_K"], "fea": th["mid_height"]["dT_epoxy_K"]})
    tm = r["thermo_mech"]["elongation_check"]
    rows.append({"check": "Conductor top axial displacement, free copper expansion", "quantity": "displacement",
                 "unit": "mm", "analytical": tm["free_copper_elongation_m"] * 1e3, "fea": tm["fea_top_uz_m"] * 1e3})
    rows.append({"check": "Conductor top axial displacement, bonded composite expansion (EA-weighted) plus free copper overhang",
                 "quantity": "displacement", "unit": "mm",
                 "analytical": tm["free_core_governed_elongation_m"] * 1e3, "fea": tm["fea_top_uz_m"] * 1e3})
    for row in rows:
        row["difference_pct"] = pct(row["fea"], row["analytical"])
    update_results("verification", rows)
    return rows


if __name__ == "__main__":
    cfg = load_config()
    out = run(cfg)
    c, m, t = out["cantilever"], out["modal"], out["thermal"]
    print(f"EI_eff = {c['EI_eff_Nm2']:.4e} N m2, EI_cond = {c['EI_conductor_Nm2']:.4e}, EI_core = {c['EI_core_Nm2']:.4e}")
    print(f"delta uniform = {c['delta_uniform_m'] * 1e3:.4f} mm, two-segment = {c['delta_two_segment_m'] * 1e3:.4f} mm")
    print(f"f1 uniform = {m['f1_uniform_Hz']:.2f} Hz, two-segment beam = {m['f1_two_segment_beam_Hz']:.2f} Hz")
    print(f"q = {t['q_W_per_m3']:.4e} W/m3, Q' = {t['Q_per_length_W_per_m']:.2f} W/m, total {t['Q_total_W']:.2f} W")
    print(f"T conductor surface {t['T_conductor_surface_C']:.2f} C, core surface {t['T_core_surface_C']:.2f} C")

### `static.py`: Stage 3: static load cases S1, S2, S3

In [ ]:
%%writefile src/static.py
"""Stage 3: linear static analysis.

S1 cantilever: horizontal force F (x direction) as a uniform traction on the top conductor end face.
S2 self-weight: gravity (-z).
S3 static-equivalent seismic: horizontal body acceleration a_h (x direction) plus gravity.
    Simplified static-equivalent check, not a seismic qualification.
Support: flange bottom face fully fixed. All bodies perfectly bonded (conforming mesh).
"""
import numpy as np
import pyvista as pv
import ufl
from dolfinx import fem, plot
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, set_bc
from petsc4py import PETSc

from common import (BODIES, RESULTS_DIR, TAGS, body_max, derived_geometry, eps, load_config,
                    material_field, nodal_max_displacement, project_dg1, sigma, singular_rings,
                    surface_average, update_results, von_mises)


class ElasticModel:
    """Assembles the stiffness matrix once (LU factorisation with MUMPS) and solves several load cases."""

    def __init__(self, mesh_data, cfg, support="fixed"):
        self.cfg = cfg
        self.msh = mesh_data.mesh
        self.ct = mesh_data.cell_tags
        self.ft = mesh_data.facet_tags
        msh = self.msh
        self.V = fem.functionspace(msh, ("Lagrange", cfg["mesh"]["element_order"], (3,)))
        self.E = material_field(msh, self.ct, cfg, "E")
        self.nu = material_field(msh, self.ct, cfg, "nu")
        self.rho = material_field(msh, self.ct, cfg, "rho")
        self.alpha = material_field(msh, self.ct, cfg, "alpha")
        self.dx = ufl.Measure("dx", msh, subdomain_data=self.ct)
        self.ds = ufl.Measure("ds", msh, subdomain_data=self.ft)
        u, v = ufl.TrialFunction(self.V), ufl.TestFunction(self.V)
        self.v = v
        fixed = self.ft.find(TAGS["fixed"])
        self.support = support
        if support == "fixed":
            zero = np.zeros(3, dtype=PETSc.ScalarType)
            self.bc = fem.dirichletbc(zero, fem.locate_dofs_topological(self.V, 2, fixed), self.V)
            self.bcs = [self.bc]
        elif support == "sliding":
            # Sensitivity case: flange bottom face restrained axially only (free radial expansion);
            # four points on the outer edge remove the rigid-body motions in x, y and rotation about z.
            self.bcs = self._sliding_bcs(fixed)
        else:
            raise ValueError(support)
        self.a = fem.form(ufl.inner(sigma(u, self.E, self.nu), eps(v)) * self.dx)
        self.A = assemble_matrix(self.a, bcs=self.bcs)
        self.A.assemble()
        self.ksp = PETSc.KSP().create(msh.comm)
        self.ksp.setOperators(self.A)
        self.ksp.setType("preonly")
        pc = self.ksp.getPC()
        pc.setType("cholesky")
        pc.setFactorSolverType("mumps")
        self.ksp.setFromOptions()
        self.ndofs = self.V.dofmap.index_map.size_global * 3

    def _sliding_bcs(self, fixed):
        d = derived_geometry(self.cfg)
        bcs = []
        dz = fem.locate_dofs_topological(self.V.sub(2), 2, fixed)
        bcs.append(fem.dirichletbc(PETSc.ScalarType(0), dz, self.V.sub(2)))
        X = self.V.tabulate_dof_coordinates()
        rf, z0 = d["r_f"], d["z_f0"]
        # ux = 0 at (0, +-rf), uy = 0 at (+-rf, 0): no net translation or rotation, radial growth free
        for comp, pts in [(0, [(0, rf, z0), (0, -rf, z0)]), (1, [(rf, 0, z0), (-rf, 0, z0)])]:
            nodes = [int(np.argmin(np.linalg.norm(X - np.array(p), axis=1))) for p in pts]
            dofs = np.array([3 * n + comp for n in nodes], dtype=np.int32)
            bcs.append(fem.dirichletbc(PETSc.ScalarType(0), dofs, self.V.sub(comp)))
        return bcs

    def solve(self, body_force=(0, 0, 0), traction_top=(0, 0, 0), dT=None):
        msh, v = self.msh, self.v
        b = fem.Constant(msh, np.array(body_force, dtype=PETSc.ScalarType))
        t = fem.Constant(msh, np.array(traction_top, dtype=PETSc.ScalarType))
        L = self.rho * ufl.dot(b, v) * self.dx + ufl.dot(t, v) * self.ds(TAGS["top_end"])
        if dT is not None:
            # thermal pre-stress term: C : (alpha dT I)
            lmbda = self.E * self.nu / ((1 + self.nu) * (1 - 2 * self.nu))
            mu = self.E / (2 * (1 + self.nu))
            L += (3 * lmbda + 2 * mu) * self.alpha * dT * ufl.div(v) * self.dx
        Lf = fem.form(L)
        rhs = assemble_vector(Lf)
        apply_lifting(rhs, [self.a], bcs=[self.bcs])
        rhs.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
        set_bc(rhs, self.bcs)
        uh = fem.Function(self.V)
        self.ksp.solve(rhs, uh.x.petsc_vec)
        uh.x.scatter_forward()
        return uh

    def von_mises(self, u, dT=None):
        s = sigma(u, self.E, self.nu, dT=dT, alpha=self.alpha if dT is not None else None)
        return project_dg1(von_mises(s), self.msh)

    def summarise(self, u, dT=None):
        cfg = self.cfg
        vm = self.von_mises(u, dT)
        rings = singular_rings(cfg)
        excl = cfg["postprocessing"]["corner_exclusion"]
        umax, uloc = nodal_max_displacement(u)
        out = {
            "max_displacement_m": umax,
            "max_displacement_location": uloc,
            "top_deflection_x_m": surface_average(u[0], self.ds(TAGS["top_end"]), self.msh),
            "top_displacement_z_m": surface_average(u[2], self.ds(TAGS["top_end"]), self.msh),
            "von_mises": {},
        }
        for body in BODIES:
            s_all, loc_all = body_max(vm, self.ct, body)
            s_ex, loc_ex = body_max(vm, self.ct, body, rings, excl)
            out["von_mises"][body] = {
                "max_Pa": s_all, "max_location": loc_all,
                "max_away_from_corners_Pa": s_ex, "away_location": loc_ex,
            }
        return out, vm


def save_fields(model, name, fields_u, fields_vm):
    """Save displacement (P2 grid) and von Mises (DG1 grid) for plotting with PyVista."""
    RESULTS_DIR.mkdir(exist_ok=True)
    grid = pv.UnstructuredGrid(*plot.vtk_mesh(model.V))
    for key, u in fields_u.items():
        grid.point_data[key] = u.x.array.reshape(-1, 3)
    grid.save(str(RESULTS_DIR / f"{name}_u.vtu"))
    if fields_vm:
        W = next(iter(fields_vm.values())).function_space
        g2 = pv.UnstructuredGrid(*plot.vtk_mesh(W))
        for key, f in fields_vm.items():
            g2.point_data[key] = f.x.array
        g2.cell_data["body"] = _cell_body(model)
        g2.save(str(RESULTS_DIR / f"{name}_vm.vtu"))


def _cell_body(model):
    n = model.msh.topology.index_map(3).size_local
    body = np.zeros(n, dtype=np.int32)
    for name in BODIES:
        body[model.ct.find(TAGS[name])] = TAGS[name]
    return body


def run(mesh_data, cfg, model=None, save=True):
    model = model or ElasticModel(mesh_data, cfg)
    d = derived_geometry(cfg)
    ld = cfg["loads"]
    A_end = np.pi * (d["r_co"] ** 2 - d["r_ci"] ** 2)
    g = ld["g"]
    cases = {
        "S1_cantilever": dict(traction_top=(ld["cantilever_force"] / A_end, 0, 0)),
        "S2_self_weight": dict(body_force=(0, 0, -g)),
        "S3_seismic_static_equivalent": dict(body_force=(ld["seismic_horizontal_g"] * g, 0, -g)),
    }
    results = {"n_dofs": model.ndofs}
    us, vms = {}, {}
    for name, kw in cases.items():
        u = model.solve(**kw)
        res, vm = model.summarise(u)
        results[name] = res
        us[name], vms[name] = u, vm
        print(f"{name}: top deflection x = {res['top_deflection_x_m'] * 1e3:.4f} mm, "
              f"max |u| = {res['max_displacement_m'] * 1e3:.4f} mm, "
              + ", ".join(f"{b} vm max {v['max_Pa'] / 1e6:.2f} MPa (away {v['max_away_from_corners_Pa'] / 1e6:.2f})"
                          for b, v in res["von_mises"].items()))
    # S1 deflection along the conductor mid-wall line in the neutral plane (x = 0), from the fixed plane up
    from common import eval_points
    r_mid = 0.5 * (d["r_ci"] + d["r_co"])
    z = np.linspace(d["z_f0"], d["z_c1"], cfg["postprocessing"]["axial_line_points"])
    pts = np.column_stack([np.zeros_like(z), np.full_like(z, r_mid), z])
    ux = eval_points(us["S1_cantilever"], pts)[:, 0]
    results["S1_cantilever"]["profile"] = {"z_m": z.tolist(), "ux_m": ux.tolist(), "line": "x = 0, y = mid-wall radius of conductor"}
    k_top = int(np.argmin(abs(z - d["L_core"])))
    results["S1_cantilever"]["ux_at_core_top_m"] = float(ux[k_top])
    # Total mass (check of self-weight)
    mass = fem.assemble_scalar(fem.form(model.rho * model.dx))
    results["total_mass_kg"] = float(mass)
    if save:
        save_fields(model, "static", us, vms)
        update_results("static", results)
    return model, us, results


if __name__ == "__main__":
    from geometry import create_mesh
    cfg = load_config()
    data, stats = create_mesh(cfg)
    run(data, cfg)

### `modal.py`: Stage 4: modal analysis with SLEPc (shift-invert)

In [ ]:
%%writefile src/modal.py
"""Stage 4: modal analysis. First natural frequencies and mode shapes with SLEPc (shift-invert about 0).

Same fixed support as the static cases (flange bottom face). Effective modal mass in x, y, z is
reported to identify each mode type.
"""
import numpy as np
import pyvista as pv
import ufl
from dolfinx import fem, plot
from dolfinx.fem.petsc import assemble_matrix
from petsc4py import PETSc
from slepc4py import SLEPc

from common import RESULTS_DIR, location, load_config, update_results


def solve_modes(model, n_modes):
    V, msh = model.V, model.msh
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    m_form = fem.form(model.rho * ufl.inner(u, v) * model.dx)
    M = assemble_matrix(m_form, bcs=model.bcs, diag=0.0)   # constrained dofs: K diag 1, M diag 0
    M.assemble()

    eps = SLEPc.EPS().create(msh.comm)
    eps.setOperators(model.A, M)
    eps.setProblemType(SLEPc.EPS.ProblemType.GHEP)
    eps.setType(SLEPc.EPS.Type.KRYLOVSCHUR)
    eps.setDimensions(nev=n_modes)
    eps.setTolerances(tol=1e-10, max_it=200)
    st = eps.getST()
    st.setType(SLEPc.ST.Type.SINVERT)
    st.setShift(0.0)
    eps.setTarget(0.0)
    eps.setWhichEigenpairs(SLEPc.EPS.Which.TARGET_MAGNITUDE)
    ksp = st.getKSP()
    ksp.setType("preonly")
    ksp.getPC().setType("cholesky")
    ksp.getPC().setFactorSolverType("mumps")
    eps.setFromOptions()
    eps.solve()
    nconv = eps.getConverged()
    assert nconv >= n_modes, f"only {nconv} modes converged"

    modes = []
    for i in range(nconv):
        lam = eps.getEigenvalue(i).real
        phi = fem.Function(V)
        eps.getEigenvector(i, phi.x.petsc_vec)
        phi.x.scatter_forward()
        err = eps.computeError(i, SLEPc.EPS.ErrorType.RELATIVE)
        modes.append((lam, phi, err))
    modes.sort(key=lambda m: m[0])
    return modes[:n_modes]


def align_degenerate_pairs(model, modes, rel_tol=1e-3):
    """Bending modes of an axisymmetric body come in pairs with (almost) equal frequency; the solver
    returns an arbitrary orthogonal combination. Rotate each pair so that the first mode moves in x
    and the second in y. Eigenvectors are mass-normalised, so the rotation keeps them orthonormal."""
    def part(phi, k):
        return fem.assemble_scalar(fem.form(model.rho * phi[k] * model.dx))
    i = 0
    while i < len(modes) - 1:
        (l1, p1, e1), (l2, p2, e2) = modes[i], modes[i + 1]
        if abs(l2 - l1) / l1 < 2 * rel_tol:
            a, b = part(p1, 0), part(p2, 0)
            n = np.hypot(a, b)
            if n > 0:
                c, s = a / n, b / n
                q1, q2 = fem.Function(model.V), fem.Function(model.V)
                q1.x.array[:] = c * p1.x.array + s * p2.x.array
                q2.x.array[:] = -s * p1.x.array + c * p2.x.array
                modes[i], modes[i + 1] = (l1, q1, e1), (l2, q2, e2)
            i += 2
        else:
            i += 1
    return modes


def mode_info(model, lam, phi, total_mass):
    msh = model.msh
    f = float(np.sqrt(lam) / (2 * np.pi))
    mod_mass = fem.assemble_scalar(fem.form(model.rho * ufl.inner(phi, phi) * model.dx))
    eff = {}
    for k, name in enumerate("xyz"):
        L = fem.assemble_scalar(fem.form(model.rho * phi[k] * model.dx))
        eff[name] = float(L ** 2 / mod_mass / total_mass)
    a = phi.x.array.reshape(-1, 3)
    X = model.V.tabulate_dof_coordinates()[: a.shape[0]]
    imax = int(np.argmax(np.linalg.norm(a, axis=1)))
    # crude classification from effective mass
    if max(eff.values()) < 0.01:
        kind = "torsion or local mode (low effective mass)"
    else:
        kdir = max(eff, key=eff.get)
        kind = f"bending in {kdir}-z plane" if kdir in "xy" else "axial"
    # mode below or above the flange: compare max amplitude location
    part = "upper part (above flange)" if X[imax, 2] > model_flange_z(model) else "lower part (below flange)"
    return {"frequency_Hz": f, "eff_mass_fraction": eff, "type": kind,
            "max_amplitude_location": location(X[imax]), "dominant_region": part,
            "modal_mass_kg": float(mod_mass)}


def model_flange_z(model):
    from common import derived_geometry
    return derived_geometry(model.cfg)["z_f0"]


def run(mesh_data, cfg, model=None, save=True):
    from static import ElasticModel
    model = model or ElasticModel(mesh_data, cfg)
    n = cfg["modal"]["n_modes"]
    modes = align_degenerate_pairs(model, solve_modes(model, n))
    total_mass = float(fem.assemble_scalar(fem.form(model.rho * model.dx)))
    out = {"n_dofs": model.ndofs, "total_mass_kg": total_mass, "modes": []}
    for i, (lam, phi, err) in enumerate(modes):
        info = mode_info(model, lam, phi, total_mass)
        info["relative_residual"] = float(err)
        out["modes"].append(info)
        print(f"mode {i + 1}: {info['frequency_Hz']:.2f} Hz  {info['type']:45s} {info['dominant_region']}  "
              f"eff mass x/y/z = {info['eff_mass_fraction']['x']:.3f}/{info['eff_mass_fraction']['y']:.3f}/"
              f"{info['eff_mass_fraction']['z']:.3f}")
    f1, f2 = out["modes"][0]["frequency_Hz"], out["modes"][1]["frequency_Hz"]
    out["f1_f2_relative_difference_pct"] = 100 * abs(f2 - f1) / f1
    if save:
        grid = pv.UnstructuredGrid(*plot.vtk_mesh(model.V))
        for i, (_, phi, _) in enumerate(modes):
            a = phi.x.array.reshape(-1, 3)
            grid.point_data[f"mode{i + 1}"] = a / np.abs(a).max()
        grid.save(str(RESULTS_DIR / "modal_u.vtu"))
        update_results("modal", out)
    return out


if __name__ == "__main__":
    from geometry import create_mesh
    cfg = load_config()
    data, _ = create_mesh(cfg)
    run(data, cfg)

### `thermal.py`: Stage 5: steady-state Joule heating

In [ ]:
%%writefile src/thermal.py
"""Stage 5: steady-state temperature from Joule heating in the conductor.

Heat source: q = I^2 rho_e / A^2 in the copper, resistivity at 20 C (no temperature dependence).
Boundary conditions:
  - convection h to T_ambient on the outer surfaces of core and flange (including the flange
    bottom face; the tank cover is not modelled),
  - conductor bore insulated,
  - exposed conductor surfaces outside the core (terminal ends) insulated, a conservative choice:
    their heat must leave through the core and flange.
Perfect thermal contact between bodies (conforming mesh, continuous temperature).
"""
import numpy as np
import pyvista as pv
import ufl
from dolfinx import fem, plot
from dolfinx.fem.petsc import LinearProblem
from mpi4py import MPI

from common import (BODIES, RESULTS_DIR, TAGS, derived_geometry, eval_points, load_config, location,
                    material_field, update_results)
from hand_calcs import joule_heating, radial_temperature


def solve(mesh_data, cfg):
    msh, ct, ft = mesh_data.mesh, mesh_data.cell_tags, mesh_data.facet_tags
    ld = cfg["loads"]
    V = fem.functionspace(msh, ("Lagrange", cfg["mesh"]["element_order"]))
    k = material_field(msh, ct, cfg, "k")
    dx = ufl.Measure("dx", msh, subdomain_data=ct)
    ds = ufl.Measure("ds", msh, subdomain_data=ft)
    q = fem.Constant(msh, joule_heating(cfg)["q_W_per_m3"])
    h = fem.Constant(msh, ld["h_conv"])
    Tinf = fem.Constant(msh, ld["T_ambient"])
    T, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    ds_conv = ds(TAGS["convection"]) + ds(TAGS["fixed"])
    a = k * ufl.inner(ufl.grad(T), ufl.grad(v)) * dx + h * T * v * ds_conv
    L = q * v * dx(TAGS["conductor"]) + h * Tinf * v * ds_conv
    problem = LinearProblem(a, L, petsc_options={"ksp_type": "preonly", "pc_type": "cholesky",
                                                 "pc_factor_mat_solver_type": "mumps"},
                            petsc_options_prefix="thermal_")
    Th = problem.solve()
    if isinstance(Th, tuple):
        Th = Th[0]
    Th.name = "T"
    # Energy balance: generated heat vs heat leaving by convection
    Q_gen = fem.assemble_scalar(fem.form(q * dx(TAGS["conductor"])))
    Q_out = fem.assemble_scalar(fem.form(h * (Th - Tinf) * ds_conv))
    return Th, float(msh.comm.allreduce(Q_gen, op=MPI.SUM)), float(msh.comm.allreduce(Q_out, op=MPI.SUM))


def run(mesh_data, cfg, save=True):
    d = derived_geometry(cfg)
    Th, Q_gen, Q_out = solve(mesh_data, cfg)
    V = Th.function_space
    X = V.tabulate_dof_coordinates()
    i = int(np.argmax(Th.x.array))
    out = {"n_dofs": int(V.dofmap.index_map.size_global),
           "heat_generated_W": Q_gen, "heat_convected_W": Q_out,
           "energy_balance_error_pct": 100 * (Q_out - Q_gen) / Q_gen,
           "T_max_C": float(Th.x.array[i]), "T_max_location": location(X[i]),
           "T_min_C": float(Th.x.array.min())}
    # Max temperature per body
    ct = mesh_data.cell_tags
    out["T_max_per_body_C"] = {}
    for b in BODIES:
        dofs = np.unique(V.dofmap.list[ct.find(TAGS[b])].flatten())
        out["T_max_per_body_C"][b] = float(Th.x.array[dofs].max())

    # Axial line: the bore is empty, so the "axis" line is taken at the conductor mid-wall radius
    pp = cfg["postprocessing"]
    r_mid = 0.5 * (d["r_ci"] + d["r_co"])
    z = np.linspace(d["z_c0"], d["z_c1"], pp["axial_line_points"])
    Tz = eval_points(Th, np.column_stack([np.full_like(z, r_mid), np.zeros_like(z), z]))[:, 0]
    z_s = np.linspace(0.0, d["L_core"], pp["axial_line_points"])
    Ts = eval_points(Th, np.column_stack([np.full_like(z_s, d["r_k"] * (1 - 1e-6)), np.zeros_like(z_s), z_s]))[:, 0]
    out["axial"] = {"line": "conductor mid-wall radius, y = 0", "z_m": z.tolist(), "T_C": Tz.tolist(),
                    "core_surface_z_m": z_s.tolist(), "core_surface_T_C": Ts.tolist()}

    # Radial line at core mid-height, compared with the 1D analytical solution
    zm = d["L_core"] / 2
    r = np.linspace(d["r_ci"] * (1 + 1e-6), d["r_k"] * (1 - 1e-6), pp["radial_line_points"])
    Tr = eval_points(Th, np.column_stack([r, np.zeros_like(r), np.full_like(r, zm)]))[:, 0]
    Ta = radial_temperature(cfg, r)
    T_co = float(eval_points(Th, np.array([[d["r_co"], 0, zm]]))[0, 0])
    T_k = float(Tr[-1])
    T_co_a, T_k_a = radial_temperature(cfg, [d["r_co"], d["r_k"]])
    out["mid_height"] = {"z_m": zm, "r_m": r.tolist(), "T_fea_C": Tr.tolist(), "T_analytical_C": Ta.tolist(),
                         "T_conductor_surface_C": T_co, "T_core_surface_C": T_k, "dT_epoxy_K": T_co - T_k,
                         "max_abs_difference_K": float(np.max(np.abs(Tr - Ta))),
                         "T_conductor_surface_analytical_C": float(T_co_a),
                         "T_core_surface_analytical_C": float(T_k_a)}
    # Local radial heat flow at mid-height (from the convective flux at the core surface) compared with
    # the generation per unit length; the 1D conduction law is then checked with this local heat flow.
    m = cfg["materials"]
    Qp_local = 2 * np.pi * d["r_k"] * cfg["loads"]["h_conv"] * (T_k - cfg["loads"]["T_ambient"])
    out["mid_height"]["Q_radial_local_W_per_m"] = float(Qp_local)
    out["mid_height"]["Q_generated_per_length_W_per_m"] = joule_heating(cfg)["Q_per_length_W_per_m"]
    out["mid_height"]["T_analytical_local_heat_flow_C"] = radial_temperature(
        cfg, r, q_scale=Qp_local / joule_heating(cfg)["Q_per_length_W_per_m"]).tolist()
    out["mid_height"]["dT_epoxy_1D_with_local_heat_flow_K"] = float(
        Qp_local / (2 * np.pi * m["epoxy"]["k"]) * np.log(d["r_k"] / d["r_co"]))
    print(f"Q generated {Q_gen:.2f} W, convected {Q_out:.2f} W; T max {out['T_max_C']:.2f} C at {out['T_max_location']}")
    print(f"mid-height: conductor surface {T_co:.2f} C (1D {T_co_a:.2f}), core surface {T_k:.2f} C (1D {T_k_a:.2f})")
    if save:
        grid = pv.UnstructuredGrid(*plot.vtk_mesh(V))
        grid.point_data["T"] = Th.x.array
        grid.save(str(RESULTS_DIR / "thermal_T.vtu"))
        update_results("thermal", out)
    return Th, out


if __name__ == "__main__":
    from geometry import create_mesh
    cfg = load_config()
    data, _ = create_mesh(cfg)
    run(data, cfg)

### `thermo_mech.py`: Stage 6: thermal stress, combination with S1, sliding-support sensitivity

In [ ]:
%%writefile src/thermo_mech.py
"""Stage 6: thermal stress from the Stage 5 temperature field.

Thermal strain alpha (T - T_ref), T_ref = 20 C (stress-free), same fixed support, no mechanical load.
The combination with S1 uses superposition of displacement fields (the model is linear).
"""
import numpy as np
import ufl
from dolfinx import fem
from mpi4py import MPI

from common import (BODIES, TAGS, body_max, derived_geometry, load_config, location, singular_rings,
                    surface_average, update_results)


def interface_max(vm, ct, body, r0, z_lo, z_hi, rings=None, excl=0.0, tol=1e-6):
    """Maximum of a DG1 field on one side (body) of a cylindrical interface r = r0, z_lo <= z <= z_hi."""
    W = vm.function_space
    dofs = np.unique(W.dofmap.list[ct.find(TAGS[body])].flatten())
    X = W.tabulate_dof_coordinates()[dofs]
    r = np.hypot(X[:, 0], X[:, 1])
    sel = (np.abs(r - r0) < 1e-5) & (X[:, 2] > z_lo - tol) & (X[:, 2] < z_hi + tol)
    if rings:
        for (rr, zz) in rings:
            sel &= np.hypot(r - rr, X[:, 2] - zz) > excl
    vals = vm.x.array[dofs][sel]
    i = int(np.argmax(vals))
    return float(vals[i]), location(X[sel][i])


def mean_over(model, f, body, z_lo, z_hi):
    x = ufl.SpatialCoordinate(model.msh)
    ind = ufl.conditional(ufl.And(ufl.ge(x[2], z_lo), ufl.le(x[2], z_hi)), 1.0, 0.0)
    num = fem.assemble_scalar(fem.form(f * ind * model.dx(TAGS[body])))
    den = fem.assemble_scalar(fem.form(ind * model.dx(TAGS[body])))
    return float(model.msh.comm.allreduce(num, op=MPI.SUM) / model.msh.comm.allreduce(den, op=MPI.SUM))


def stress_summary(model, u, dT, cfg):
    d = derived_geometry(cfg)
    rings = singular_rings(cfg)
    excl = cfg["postprocessing"]["corner_exclusion"]
    vm = model.von_mises(u, dT)
    out = {"von_mises": {}, "interfaces": {}}
    for b in BODIES:
        s, loc = body_max(vm, model.ct, b)
        s2, loc2 = body_max(vm, model.ct, b, rings, excl)
        out["von_mises"][b] = {"max_Pa": s, "max_location": loc, "max_away_from_corners_Pa": s2, "away_location": loc2}
    ifs = {"flange_to_core": (d["r_k"], d["z_f0"], d["z_f1"], ["core", "flange"]),
           "conductor_to_core": (d["r_co"], 0.0, d["L_core"], ["conductor", "core"])}
    for name, (r0, z0, z1, bodies) in ifs.items():
        out["interfaces"][name] = {}
        for b in bodies:
            s, loc = interface_max(vm, model.ct, b, r0, z0, z1)
            s2, loc2 = interface_max(vm, model.ct, b, r0, z0, z1, rings, excl)
            out["interfaces"][name][b] = {"max_Pa": s, "max_location": loc,
                                          "max_away_from_corners_Pa": s2, "away_location": loc2}
    return out, vm


def run(mesh_data, cfg, T, model=None, u_s1=None, save=True):
    from static import ElasticModel, save_fields
    model = model or ElasticModel(mesh_data, cfg)
    d = derived_geometry(cfg)
    ld = cfg["loads"]
    m = cfg["materials"]
    dT = T - ld["T_reference"]
    u_th = model.solve(dT=dT)
    out = {"T_reference_C": ld["T_reference"]}
    th, vm_th = stress_summary(model, u_th, dT, cfg)
    th["top_displacement_z_m"] = surface_average(u_th[2], model.ds(TAGS["top_end"]), model.msh)
    th["top_displacement_x_m"] = surface_average(u_th[0], model.ds(TAGS["top_end"]), model.msh)
    out["thermal_only"] = th

    # Combination with S1 by superposition of displacements
    if u_s1 is None:
        A_end = np.pi * (d["r_co"] ** 2 - d["r_ci"] ** 2)
        u_s1 = model.solve(traction_top=(ld["cantilever_force"] / A_end, 0, 0))
    u_c = fem.Function(model.V)
    u_c.x.array[:] = u_th.x.array + u_s1.x.array
    comb, vm_c = stress_summary(model, u_c, dT, cfg)
    comb["top_deflection_x_m"] = surface_average(u_c[0], model.ds(TAGS["top_end"]), model.msh)
    out["thermal_plus_S1"] = comb

    # Sanity check: axial elongation from the fixed plane to the conductor top
    a = d["L_core"] - d["z_f0"]
    b = d["z_c1"] - d["L_core"]
    L = a + b
    dT_cond = mean_over(model, dT, "conductor", d["z_f0"], d["z_c1"])
    dT_cond_seg = mean_over(model, dT, "conductor", d["z_f0"], d["L_core"])
    dT_core_seg = mean_over(model, dT, "core", d["z_f0"], d["L_core"])
    dT_over = mean_over(model, dT, "conductor", d["L_core"], d["z_c1"])
    free_cu = m["copper"]["alpha"] * L * dT_cond
    EA_c = m["copper"]["E"] * np.pi * (d["r_co"] ** 2 - d["r_ci"] ** 2)
    EA_k = m["epoxy"]["E"] * np.pi * (d["r_k"] ** 2 - d["r_co"] ** 2)
    strain_seg = (EA_c * m["copper"]["alpha"] * dT_cond_seg + EA_k * m["epoxy"]["alpha"] * dT_core_seg) / (EA_c + EA_k)
    composite = strain_seg * a + m["copper"]["alpha"] * dT_over * b
    out["elongation_check"] = {
        "L_m": L, "mean_dT_conductor_K": dT_cond, "mean_dT_conductor_core_segment_K": dT_cond_seg,
        "mean_dT_core_segment_K": dT_core_seg, "mean_dT_conductor_overhang_K": dT_over,
        "free_copper_elongation_m": free_cu,
        "free_core_governed_elongation_m": composite,
        "fea_top_uz_m": th["top_displacement_z_m"],
        "note": "composite estimate: EA-weighted free strain of the bonded core segment plus free copper overhang",
    }
    # Sensitivity: same thermal load with a sliding support (axial restraint only at the flange face)
    sl = ElasticModel(mesh_data, cfg, support="sliding")
    u_sl = sl.solve(dT=dT)
    sl_sum, vm_sl = stress_summary(sl, u_sl, dT, cfg)
    out["thermal_only_sliding_support"] = sl_sum
    print("sliding support: " + ", ".join(f"{k} {v['max_Pa'] / 1e6:.2f} MPa (away {v['max_away_from_corners_Pa'] / 1e6:.2f}) at z {v['away_location']['z']}"
                                          for k, v in sl_sum["von_mises"].items()))
    print(f"thermal only: " + ", ".join(f"{k} {v['max_Pa'] / 1e6:.2f} MPa (away {v['max_away_from_corners_Pa'] / 1e6:.2f})"
                                       for k, v in th["von_mises"].items()))
    print(f"thermal + S1: " + ", ".join(f"{k} {v['max_Pa'] / 1e6:.2f} MPa (away {v['max_away_from_corners_Pa'] / 1e6:.2f})"
                                       for k, v in comb["von_mises"].items()))
    print(f"top uz FEA {th['top_displacement_z_m'] * 1e3:.4f} mm, free copper {free_cu * 1e3:.4f} mm, composite {composite * 1e3:.4f} mm")
    if save:
        save_fields(model, "thermo_mech", {"u_thermal": u_th, "u_thermal_plus_S1": u_c},
                    {"vm_thermal": vm_th, "vm_thermal_plus_S1": vm_c, "vm_thermal_sliding": vm_sl})
        update_results("thermo_mech", out)
    return out


if __name__ == "__main__":
    from geometry import create_mesh
    import thermal
    cfg = load_config()
    data, _ = create_mesh(cfg)
    T, _ = thermal.run(data, cfg, save=False)
    run(data, cfg, T)

### `convergence.py`: Stage 7: mesh convergence

In [ ]:
%%writefile src/convergence.py
"""Stage 7: mesh convergence study for S1 (top deflection, maximum stress) and f1, on three meshes."""
import gc

from common import load_config, update_results
from geometry import create_mesh
from hand_calcs import sections
from common import pct_change


def run(cfg):
    from modal import solve_modes
    from static import ElasticModel

    A_end = sections(cfg)["A_c"]
    rows = []
    for h in cfg["mesh"]["convergence_h"]:
        data, stats = create_mesh(cfg, h=h)
        model = ElasticModel(data, cfg)
        u = model.solve(traction_top=(cfg["loads"]["cantilever_force"] / A_end, 0, 0))
        s1, _ = model.summarise(u)
        model.ksp.destroy()          # free the static factorisation before the modal solve (memory)
        gc.collect()
        modes = solve_modes(model, 2)
        f1 = float(modes[0][0] ** 0.5 / (2 * 3.141592653589793))
        vm = s1["von_mises"]
        row = {"h_m": h, "n_elements": stats["n_elements"], "n_nodes": stats["n_nodes"], "n_dofs": model.ndofs,
               "quality_minSICN_min": stats["quality_minSICN_min"],
               "top_deflection_m": s1["top_deflection_x_m"],
               "max_von_mises_Pa": max(v["max_Pa"] for v in vm.values()),
               "max_von_mises_body": max(vm, key=lambda b: vm[b]["max_Pa"]),
               "max_von_mises_away_from_corners_Pa": max(v["max_away_from_corners_Pa"] for v in vm.values()),
               "von_mises_per_body": {b: {"max_Pa": v["max_Pa"], "away_Pa": v["max_away_from_corners_Pa"]}
                                      for b, v in vm.items()},
               "f1_Hz": f1}
        rows.append(row)
        print(f"h = {h * 1e3:.1f} mm: {row['n_elements']} elements, {row['n_dofs']} dofs, "
              f"top deflection {row['top_deflection_m'] * 1e3:.4f} mm, max vm {row['max_von_mises_Pa'] / 1e6:.2f} MPa "
              f"(away {row['max_von_mises_away_from_corners_Pa'] / 1e6:.2f}), f1 {f1:.3f} Hz", flush=True)
        del model, modes, u, data
        gc.collect()
    m, f = rows[-2], rows[-1]
    change = {
        "top_deflection_pct": pct_change(f["top_deflection_m"], m["top_deflection_m"]),
        "max_von_mises_pct": pct_change(f["max_von_mises_Pa"], m["max_von_mises_Pa"]),
        "max_von_mises_away_from_corners_pct": pct_change(f["max_von_mises_away_from_corners_Pa"],
                                                           m["max_von_mises_away_from_corners_Pa"]),
        "f1_pct": pct_change(f["f1_Hz"], m["f1_Hz"]),
    }
    per_body = {b: {"max_pct": pct_change(f["von_mises_per_body"][b]["max_Pa"], m["von_mises_per_body"][b]["max_Pa"]),
                    "away_pct": pct_change(f["von_mises_per_body"][b]["away_Pa"], m["von_mises_per_body"][b]["away_Pa"])}
                for b in f["von_mises_per_body"]}
    out = {"rows": rows, "change_medium_to_fine": change, "change_medium_to_fine_per_body": per_body}
    update_results("convergence", out)
    print("medium to fine change:", {k: round(v, 3) for k, v in change.items()})
    return out


if __name__ == "__main__":
    run(load_config())

### `make_figures.py`: Stage 8: figures

In [ ]:
%%writefile src/make_figures.py
"""Stage 8: figures (PNG, 300 dpi). Fields are cut on a meridian plane with PyVista and drawn with
Matplotlib (no OpenGL needed)."""
import json

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import numpy as np
import pyvista as pv
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

from common import FIGURES_DIR, RESULTS_DIR, TAGS, derived_geometry, load_config, read_results
from hand_calcs import cantilever_profile

# Palette: categorical slots 1-3 and a single-hue sequential ramp (light to dark blue)
C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#d9d8d4"
SEQ = LinearSegmentedColormap.from_list(
    "seq_blue", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
BODY_COLOURS = {"conductor": C2, "core": C1, "flange": C3}
DPI = 300

plt.rcParams.update({
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9, "axes.edgecolor": INK2,
    "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False,
    "savefig.dpi": DPI, "savefig.bbox": "tight", "legend.frameon": False,
})


def cut(grid, normal="y"):
    """Slice on the plane through the axis; returns a triangulated PolyData."""
    s = grid.slice(normal=normal, origin=(0, 0, 0)).triangulate()
    return s


def tri_of(s, normal="y", disp=None, scale=0.0):
    P = np.array(s.points)
    if disp is not None:
        P = P + scale * np.asarray(s.point_data[disp])
    h = P[:, 0] if normal == "y" else P[:, 1]
    faces = s.faces.reshape(-1, 4)[:, 1:]
    return mtri.Triangulation(h, P[:, 2], faces)


def field_plot(ax, s, values, normal="y", disp=None, scale=0.0, cmap=SEQ, vmin=None, vmax=None, edges=False):
    tri = tri_of(s, normal, disp, scale)
    tpc = ax.tripcolor(tri, values, shading="gouraud", cmap=cmap, vmin=vmin, vmax=vmax, rasterized=True)
    if edges:
        ax.triplot(tri, color="white", lw=0.15, alpha=0.6)
    ax.set_aspect("equal")
    return tpc


def outline(ax, cfg, normal_sign=1, colour=INK2, lw=0.6):
    """Undeformed outline of the three bodies in the r-z half planes (both sides of the axis)."""
    d = derived_geometry(cfg)
    for sgn in (1, -1):
        rects = [(d["r_ci"], d["z_c0"], d["r_co"] - d["r_ci"], d["L_cond"]),
                 (d["r_co"], 0.0, d["r_k"] - d["r_co"], d["L_core"]),
                 (d["r_k"], d["z_f0"], d["r_f"] - d["r_k"], d["t_f"])]
        for (r0, z0, w, hh) in rects:
            xs = sgn * np.array([r0, r0 + w, r0 + w, r0, r0])
            ax.plot(xs, [z0, z0, z0 + hh, z0 + hh, z0], color=colour, lw=lw, ls="--")


def colourbar(fig, mappable, ax, label):
    cb = fig.colorbar(mappable, ax=ax, shrink=0.85, pad=0.02)
    cb.set_label(label)
    cb.outline.set_visible(False)
    return cb


def body_values_on_cut(s):
    return np.asarray(s.cell_data["body"]) if "body" in s.cell_data else None


def fig_geometry(cfg, res):
    g = pv.read(RESULTS_DIR / "static_vm.vtu")
    s = cut(g)
    tri = tri_of(s)
    body = np.asarray(s.cell_data["body"])
    colours = {TAGS[b]: c for b, c in BODY_COLOURS.items()}
    fc = np.array([TAGS[b] for b in BODY_COLOURS])
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 6.2), gridspec_kw={"width_ratios": [1, 1.35]})
    for ax, (xl, zl) in zip(axes, [((-0.17, 0.17), (-0.25, 1.45)), ((0.0, 0.17), (0.33, 0.50))]):
        for tag in fc:
            m = body == tag
            ax.tripcolor(mtri.Triangulation(tri.x, tri.y, tri.triangles[m]),
                         facecolors=np.ones(m.sum()), cmap=matplotlib.colors.ListedColormap([colours[tag]]),
                         edgecolors="none", rasterized=True)
        ax.triplot(tri, color="white", lw=0.12 if ax is axes[0] else 0.3)
        ax.set_xlim(*xl)
        ax.set_ylim(*zl)
        ax.set_aspect("equal")
        ax.set_xlabel("x (m)")
        ax.set_ylabel("z (m)")
    d = derived_geometry(cfg)
    axes[0].annotate("fixed support\n(flange\nbottom face)", xy=(0.12, d["z_f0"]), xytext=(0.19, 0.15),
                     fontsize=7, arrowprops=dict(arrowstyle="-", color=INK2, lw=0.6), annotation_clip=False)
    axes[0].annotate("terminal\nload F", xy=(0.02, d["z_c1"]), xytext=(0.19, 1.30), fontsize=7,
                     arrowprops=dict(arrowstyle="-", color=INK2, lw=0.6), annotation_clip=False)
    axes[0].set_xticks([-0.1, 0.1])
    axes[0].set_title("Axial cut (y = 0)")
    axes[1].set_title("Mesh cut at the flange")
    handles = [Patch(color=c, label=f"{b} ({m})") for (b, c), m in
               zip(BODY_COLOURS.items(), ["copper", "epoxy", "aluminium"])]
    fig.legend(handles=handles, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.02))
    m = res["mesh"]
    fig.suptitle(f"Simplified bushing model: {m['n_elements']:,} quadratic tetrahedra, h = {m['h_global_m'] * 1e3:.0f} mm",
                 fontsize=10)
    fig.savefig(FIGURES_DIR / "fig1_geometry_mesh.png")
    plt.close(fig)


def fig_s1_displacement(cfg, res):
    g = pv.read(RESULTS_DIR / "static_u.vtu")
    s = cut(g)
    u = np.asarray(s.point_data["S1_cantilever"])
    mag = np.linalg.norm(u, axis=1) * 1e3
    scale = 50.0
    fig, ax = plt.subplots(figsize=(4.2, 6.2))
    tpc = field_plot(ax, s, mag, disp="S1_cantilever", scale=scale)
    outline(ax, cfg)
    colourbar(fig, tpc, ax, "displacement magnitude (mm)")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("z (m)")
    r = res["static"]["S1_cantilever"]
    ax.set_title(f"S1 cantilever, F = {cfg['loads']['cantilever_force']:.0f} N\n"
                 f"deformation x{scale:.0f}, top deflection {r['top_deflection_x_m'] * 1e3:.3f} mm", fontsize=9)
    fig.savefig(FIGURES_DIR / "fig2_s1_displacement.png")
    plt.close(fig)


def fig_s1_stress(cfg, res):
    g = pv.read(RESULTS_DIR / "static_vm.vtu")
    s = cut(g)
    vm = np.asarray(s.point_data["S1_cantilever"]) / 1e6
    d = derived_geometry(cfg)
    fig, axes = plt.subplots(1, 2, figsize=(7.6, 5.2), gridspec_kw={"width_ratios": [0.55, 1.45]})
    P = np.array(s.points)
    for ax, (xl, zl), lab in zip(axes, [((-0.17, 0.17), (-0.25, 1.45)), ((0.0, 0.16), (0.30, 0.56))],
                                 ["von Mises (MPa)", "von Mises (MPa), zoom scale"]):
        inside = (P[:, 0] >= xl[0]) & (P[:, 0] <= xl[1]) & (P[:, 2] >= zl[0]) & (P[:, 2] <= zl[1])
        tpc = field_plot(ax, s, vm, vmin=0, vmax=float(np.percentile(vm[inside], 99.9)))
        ax.set_xlim(*xl)
        ax.set_ylim(*zl)
        ax.set_xlabel("x (m)")
        ax.set_ylabel("z (m)")
        colourbar(fig, tpc, ax, lab)
    axes[0].set_xticks([-0.1, 0.1])
    outline(axes[1], cfg, lw=0.5)
    for (rr, zz) in [(d["r_k"], d["z_f0"]), (d["r_k"], d["z_f1"])]:
        axes[1].add_patch(plt.Circle((rr, zz), cfg["postprocessing"]["corner_exclusion"], fill=False,
                                     ec=C2, lw=0.8, ls=":"))
    axes[0].set_title("Whole bushing")
    axes[1].set_title("Zoom at the flange (tension side, x > 0)")
    axes[1].text(0.005, 0.305, "dotted circles: corner exclusion radius", color=INK2, fontsize=7)
    fig.suptitle("S1 cantilever: von Mises stress on the axial cut (y = 0)", fontsize=10)
    fig.savefig(FIGURES_DIR / "fig3_s1_von_mises_flange.png")
    plt.close(fig)


def fig_modes(cfg, res):
    g = pv.read(RESULTS_DIR / "modal_u.vtu")
    fig, axes = plt.subplots(1, 2, figsize=(6.4, 6.0), sharey=True)
    modes = res["modal"]["modes"]
    for k, (ax, normal) in enumerate(zip(axes, ["y", "x"])):
        s = cut(g, normal)
        key = f"mode{k + 1}"
        u = np.asarray(s.point_data[key])
        mag = np.linalg.norm(u, axis=1)
        tpc = field_plot(ax, s, mag, normal=normal, disp=key, scale=0.12, vmin=0, vmax=1)
        outline(ax, cfg)
        ax.set_xlabel("x (m)" if normal == "y" else "y (m)")
        f = modes[k]["frequency_Hz"]
        ax.set_title(f"Mode {k + 1}: {f:.2f} Hz\n{modes[k]['type']}", fontsize=9)
    axes[0].set_ylabel("z (m)")
    colourbar(fig, tpc, axes[1], "normalised displacement (-)")
    fig.suptitle(f"First two mode shapes (difference {res['modal']['f1_f2_relative_difference_pct']:.3f} %)",
                 fontsize=10)
    fig.savefig(FIGURES_DIR / "fig4_mode_shapes.png")
    plt.close(fig)


def fig_thermal(cfg, res):
    g = pv.read(RESULTS_DIR / "thermal_T.vtu")
    s = cut(g)
    T = np.asarray(s.point_data["T"])
    th = res["thermal"]
    fig, axes = plt.subplots(1, 2, figsize=(7.8, 5.4), gridspec_kw={"width_ratios": [0.9, 1.3], "wspace": 0.55})
    tpc = field_plot(axes[0], s, T)
    axes[0].set_xlim(-0.17, 0.17)
    axes[0].set_xlabel("x (m)")
    axes[0].set_ylabel("z (m)")
    axes[0].set_title(f"Temperature, I = {cfg['loads']['current']:.0f} A\nhot spot {th['T_max_C']:.1f} C")
    colourbar(fig, tpc, axes[0], "temperature (C)")
    mh = th["mid_height"]
    r = np.array(mh["r_m"]) * 1e3
    ax = axes[1]
    ax.plot(r, mh["T_fea_C"], color=C1, lw=2, label="FEA")
    ax.plot(r, mh["T_analytical_C"], color=C2, lw=2, ls="--", label="1D, local generation")
    ax.plot(r, mh["T_analytical_local_heat_flow_C"], color=C3, lw=2, ls=":", label="1D, FEA local heat flow")
    d = derived_geometry(cfg)
    for rr in (d["r_co"], d["r_k"]):
        ax.axvline(rr * 1e3, color=GRID, lw=0.8, zorder=0)
    ax.text(d["r_co"] * 1e3 + 1, ax.get_ylim()[0] + 1, "copper | epoxy", fontsize=7, color=INK2)
    ax.set_xlabel("radius r (mm)")
    ax.set_ylabel("temperature (C)")
    ax.set_title(f"Radial profile at core mid-height, z = {mh['z_m']:.2f} m")
    ax.legend(loc="upper right")
    ax.grid(axis="y", color=GRID, lw=0.5)
    fig.savefig(FIGURES_DIR / "fig5_temperature.png")
    plt.close(fig)


def fig_thermal_stress(cfg, res):
    g = pv.read(RESULTS_DIR / "thermo_mech_vm.vtu")
    s = cut(g)
    a = np.asarray(s.point_data["vm_thermal"]) / 1e6
    b = np.asarray(s.point_data["vm_thermal_sliding"]) / 1e6
    vmax = np.percentile(np.concatenate([a, b]), 99.9)
    fig, axes = plt.subplots(1, 2, figsize=(7.4, 4.4), sharey=True)
    for ax, v, t in zip(axes, [a, b], ["fixed flange face (baseline)", "sliding flange face (sensitivity)"]):
        tpc = field_plot(ax, s, v, vmin=0, vmax=vmax)
        ax.set_xlim(0.0, 0.16)
        ax.set_ylim(0.30, 0.56)
        outline(ax, cfg, lw=0.5)
        ax.set_xlabel("x (m)")
        ax.set_title(t)
    axes[0].set_ylabel("z (m)")
    colourbar(fig, tpc, axes[1], "von Mises stress (MPa)")
    fig.suptitle(f"Thermal stress near the flange (T_ref = {cfg['loads']['T_reference']:.0f} C, no mechanical load)",
                 fontsize=10)
    fig.savefig(FIGURES_DIR / "fig6_thermal_stress_flange.png")
    plt.close(fig)


def fig_convergence(cfg, res):
    rows = res["convergence"]["rows"]
    n = np.array([r["n_elements"] for r in rows])
    fig, axes = plt.subplots(1, 3, figsize=(7.6, 2.8))
    panels = [
        ([np.array([r["top_deflection_m"] for r in rows]) * 1e3], ["top deflection"], "S1 top deflection (mm)"),
        ([np.array([r["max_von_mises_Pa"] for r in rows]) / 1e6,
          np.array([r["max_von_mises_away_from_corners_Pa"] for r in rows]) / 1e6],
         ["maximum (at a corner)", "maximum away from corners"], "S1 von Mises stress (MPa)"),
        ([np.array([r["f1_Hz"] for r in rows])], ["f1"], "first natural frequency (Hz)"),
    ]
    for ax, (ys, labels, yl) in zip(axes, panels):
        for y, lab, c in zip(ys, labels, [C1, C2]):
            ax.plot(n, y, color=c, lw=2, marker="o", ms=5, label=lab)
        ax.set_xscale("log")
        ax.set_xlabel("number of elements")
        ax.set_ylabel(yl)
        ax.grid(axis="y", color=GRID, lw=0.5)
        if len(ys) > 1:
            ax.legend(fontsize=7, loc="best")
        ax.set_xticks(n)
        ax.set_xticklabels([f"{v / 1e3:.0f}k" for v in n])
        ax.minorticks_off()
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "fig7_mesh_convergence.png")
    plt.close(fig)


def fig_deflection_profile(cfg, res):
    p = res["static"]["S1_cantilever"]["profile"]
    d = derived_geometry(cfg)
    z = np.array(p["z_m"])
    s_pos = z - d["z_f0"]
    fig, ax = plt.subplots(figsize=(4.6, 3.2))
    ax.plot(z, np.array(p["ux_m"]) * 1e3, color=C1, lw=2, label="FEA (conductor mid-wall, neutral plane)")
    ax.plot(z, cantilever_profile(cfg, s_pos) * 1e3, color=C2, lw=2, ls="--", label="two-segment beam")
    ax.axvline(d["L_core"], color=GRID, lw=0.8, zorder=0)
    ax.text(d["L_core"] - 0.01, 0.05, "core top", ha="right", fontsize=7, color=INK2)
    ax.set_xlabel("height z (m)")
    ax.set_ylabel("deflection u_x (mm)")
    ax.set_title("S1 deflection along the bushing above the flange")
    ax.legend(loc="upper left", fontsize=7)
    ax.grid(axis="y", color=GRID, lw=0.5)
    fig.savefig(FIGURES_DIR / "fig8_s1_deflection_profile.png")
    plt.close(fig)


def main():
    FIGURES_DIR.mkdir(exist_ok=True)
    cfg = load_config()
    res = read_results()
    for f in [fig_geometry, fig_s1_displacement, fig_s1_stress, fig_modes, fig_thermal, fig_thermal_stress,
              fig_deflection_profile]:
        f(cfg, res)
        print("written", f.__name__)
    if "convergence" in res:
        fig_convergence(cfg, res)
        print("written fig_convergence")


if __name__ == "__main__":
    main()

### `make_tables.py`: Stages 9 and 10: report tables and generated documents

In [ ]:
%%writefile src/make_tables.py
"""Generate every number used in the report, README, explainer and CV entry from results/results.json.

- report/generated/values.tex : one macro per value, used in report.tex as \\V{key}
- report/generated/*.tex      : tables
- README.md, docs/EXPLAINER.md, docs/CV_ENTRY.md : filled from docs/templates/*.md ({{key}} placeholders)

A placeholder without a value stops the script, so no hand-typed number can slip in.
"""
import datetime as dt
import re

from common import ROOT, read_results

GEN = ROOT / "report" / "generated"
TEMPLATES = ROOT / "docs" / "templates"


def f(x, nd=2):
    return f"{x:,.{nd}f}"


def values(r):
    V = {}
    inp, g = r["inputs"], r["geometry"]
    ld, mt = inp["loads"], inp["materials"]
    # inputs and geometry
    V["F-N"] = f(ld["cantilever_force"], 0)
    V["seis-g"] = f(ld["seismic_horizontal_g"], 1)
    V["I-A"] = f(ld["current"], 0)
    V["h-conv"] = f(ld["h_conv"], 0)
    V["T-amb"] = f(ld["T_ambient"], 0)
    V["T-ref"] = f(ld["T_reference"], 0)
    V["g"] = f(ld["g"], 2)
    for k, nd in [("r_ci", 0), ("r_co", 0), ("r_k", 0), ("r_f", 0), ("t_f", 0)]:
        V[f"{k.replace('_', '')}-mm"] = f(g[k] * 1e3, nd)
    for k in ["L_core", "L_cond", "z_f0", "z_f1", "z_c0", "z_c1"]:
        V[f"{k.replace('_', '')}-m"] = f(g[k], 2)
    V["Dco-mm"] = f(2 * g["r_co"] * 1e3, 0)
    V["Dci-mm"] = f(2 * g["r_ci"] * 1e3, 0)
    V["Dk-mm"] = f(2 * g["r_k"] * 1e3, 0)
    V["Df-mm"] = f(2 * g["r_f"] * 1e3, 0)
    V["excl-mm"] = f(inp["postprocessing"]["corner_exclusion"] * 1e3, 0)
    V["n-modes"] = str(inp["modal"]["n_modes"])
    # software
    for k, v in r["software"].items():
        V[f"ver-{k}"] = v
    # mesh
    m = r["mesh"]
    V["mesh-h-mm"] = f(m["h_global_m"] * 1e3, 0)
    V["mesh-hc-mm"] = f(m["h_conductor_m"] * 1e3, 0)
    V["mesh-hf-mm"] = f(m["h_flange_m"] * 1e3, 0)
    V["mesh-nel"] = f(m["n_elements"], 0)
    V["mesh-nnodes"] = f(m["n_nodes"], 0)
    V["mesh-qmin"] = f(m["quality_minSICN_min"], 2)
    V["mesh-qmean"] = f(m["quality_minSICN_mean"], 2)
    V["mesh-gmin"] = f(m["quality_gamma_min"], 2)
    V["ndofs"] = f(r["static"]["n_dofs"], 0)
    V["mass-kg"] = f(r["static"]["total_mass_kg"], 1)
    # hand calcs
    hc = r["hand_calcs"]
    c = hc["cantilever"]
    V["EIc"] = f"{c['EI_conductor_Nm2']:.3g}"
    V["EIk"] = f"{c['EI_core_Nm2']:.3g}"
    V["EIeff"] = f"{c['EI_eff_Nm2']:.3g}"
    V["L-beam-m"] = f(c["L_m"], 2)
    V["a-beam-m"] = f(c["a_core_segment_m"], 2)
    V["b-beam-m"] = f(c["b_bare_conductor_m"], 2)
    V["meff"] = f(hc["modal"]["m_eff_kg_per_m"], 1)
    V["mc"] = f(hc["sections"]["m_c"], 2)
    V["sig-cond-beam-MPa"] = f(c["bending_stress_conductor_at_core_top_Pa"] / 1e6, 1)
    p = c["two_segment_parts_m"]
    V["seg1-mm"] = f(p["segment1_deflection"] * 1e3, 3)
    V["seg1rot-mm"] = f(p["segment1_rotation_times_b"] * 1e3, 3)
    V["seg2-mm"] = f(p["segment2_cantilever"] * 1e3, 3)
    V["share-overhang-pct"] = f(100 * (p["segment1_rotation_times_b"] + p["segment2_cantilever"]) /
                                c["delta_two_segment_m"], 0)
    th_hc = hc["thermal"]
    V["q-Wm3"] = f"{th_hc['q_W_per_m3']:.3g}"
    V["Qp-Wm"] = f(th_hc["Q_per_length_W_per_m"], 1)
    V["Qtot-W"] = f(th_hc["Q_total_W"], 1)
    # static
    st = r["static"]
    for case, key in [("S1_cantilever", "S1"), ("S2_self_weight", "S2"), ("S3_seismic_static_equivalent", "S3")]:
        s = st[case]
        V[f"{key}-top-mm"] = f(s["top_deflection_x_m"] * 1e3, 3)
        V[f"{key}-umax-mm"] = f(s["max_displacement_m"] * 1e3, 3)
        V[f"{key}-umax-z"] = f(s["max_displacement_location"]["z"], 2)
        for b, v in s["von_mises"].items():
            V[f"{key}-{b}-max"] = f(v["max_Pa"] / 1e6, 2)
            V[f"{key}-{b}-away"] = f(v["max_away_from_corners_Pa"] / 1e6, 2)
            V[f"{key}-{b}-maxz"] = f(v["max_location"]["z"], 2)
            V[f"{key}-{b}-awayz"] = f(v["away_location"]["z"], 2)
    V["S1-uxcoretop-mm"] = f(st["S1_cantilever"]["ux_at_core_top_m"] * 1e3, 3)
    V["S2-umax-um"] = f(st["S2_self_weight"]["max_displacement_m"] * 1e6, 2)
    V["S3-top-um"] = f(st["S3_seismic_static_equivalent"]["top_deflection_x_m"] * 1e6, 1)
    # modal
    md = r["modal"]
    for i, mode in enumerate(md["modes"], 1):
        V[f"f{i}"] = f(mode["frequency_Hz"], 2)
        V[f"f{i}-effx"] = f(100 * mode["eff_mass_fraction"]["x"], 1)
        V[f"f{i}-effy"] = f(100 * mode["eff_mass_fraction"]["y"], 1)
    V["f12-diff-pct"] = f(md["f1_f2_relative_difference_pct"], 3)
    V["f1-uniform"] = f(hc["modal"]["f1_uniform_Hz"], 1)
    V["n-beam-el"] = str(hc["modal"]["n_beam_elements"])
    V["f1-twoseg"] = f(hc["modal"]["f1_two_segment_beam_Hz"], 1)
    # thermal
    th = r["thermal"]
    V["Tmax"] = f(th["T_max_C"], 1)
    V["Tmax-z"] = f(th["T_max_location"]["z"], 2)
    V["Tmax-rise"] = f(th["T_max_C"] - ld["T_ambient"], 1)
    for b, v in th["T_max_per_body_C"].items():
        V[f"Tmax-{b}"] = f(v, 1)
    V["Q-gen"] = f(th["heat_generated_W"], 2)
    V["Q-out"] = f(th["heat_convected_W"], 2)
    V["energy-err"] = f"{abs(th['energy_balance_error_pct']):.1e}"
    mh = th["mid_height"]
    V["mid-z"] = f(mh["z_m"], 2)
    V["mid-Tco-fea"] = f(mh["T_conductor_surface_C"], 1)
    V["mid-Tk-fea"] = f(mh["T_core_surface_C"], 1)
    V["mid-Tco-an"] = f(mh["T_conductor_surface_analytical_C"], 1)
    V["mid-Tk-an"] = f(mh["T_core_surface_analytical_C"], 1)
    V["mid-dT-fea"] = f(mh["dT_epoxy_K"], 2)
    V["mid-dT-local"] = f(mh["dT_epoxy_1D_with_local_heat_flow_K"], 2)
    V["mid-Qlocal"] = f(mh["Q_radial_local_W_per_m"], 1)
    V["mid-maxdiff"] = f(mh["max_abs_difference_K"], 1)
    V["mid-Qlocal-excess-pct"] = f(100 * (mh["Q_radial_local_W_per_m"] / mh["Q_generated_per_length_W_per_m"] - 1), 0)
    V["dT-epoxy-an"] = f(th_hc["dT_epoxy_K"], 2)
    # thermal-structural
    tm = r["thermo_mech"]
    for case, key in [("thermal_only", "TH"), ("thermal_plus_S1", "THS1"), ("thermal_only_sliding_support", "THSL")]:
        for b, v in tm[case]["von_mises"].items():
            V[f"{key}-{b}-max"] = f(v["max_Pa"] / 1e6, 1)
            V[f"{key}-{b}-away"] = f(v["max_away_from_corners_Pa"] / 1e6, 1)
            V[f"{key}-{b}-maxz"] = f(v["max_location"]["z"], 2)
            V[f"{key}-{b}-maxr"] = f(v["max_location"]["r"] * 1e3, 0)
        for iface, sides in tm[case]["interfaces"].items():
            for b, v in sides.items():
                V[f"{key}-{iface}-{b}-max"] = f(v["max_Pa"] / 1e6, 1)
                V[f"{key}-{iface}-{b}-away"] = f(v["max_away_from_corners_Pa"] / 1e6, 1)
    V["TH-uz-mm"] = f(tm["thermal_only"]["top_displacement_z_m"] * 1e3, 3)
    V["THS1-top-mm"] = f(tm["thermal_plus_S1"]["top_deflection_x_m"] * 1e3, 3)
    el = tm["elongation_check"]
    V["el-dTcond"] = f(el["mean_dT_conductor_K"], 1)
    V["el-dTcondseg"] = f(el["mean_dT_conductor_core_segment_K"], 1)
    V["el-dTcore"] = f(el["mean_dT_core_segment_K"], 1)
    V["el-dTover"] = f(el["mean_dT_conductor_overhang_K"], 1)
    V["el-free-mm"] = f(el["free_copper_elongation_m"] * 1e3, 3)
    V["el-comp-mm"] = f(el["free_core_governed_elongation_m"] * 1e3, 3)
    V["el-fea-mm"] = f(el["fea_top_uz_m"] * 1e3, 3)
    # verification rows
    for i, row in enumerate(r["verification"], 1):
        nd = 3 if row["unit"] == "mm" else 2
        V[f"ver{i}-an"] = f(row["analytical"], nd)
        V[f"ver{i}-fea"] = f(row["fea"], nd)
        V[f"ver{i}-diff"] = f"{row['difference_pct']:+.1f}"
        V[f"ver{i}-absdiff"] = f"{abs(row['difference_pct']):.1f}"
    # convergence
    if "convergence" in r:
        cv = r["convergence"]
        for i, row in enumerate(cv["rows"]):
            k = ["coarse", "medium", "fine"][i]
            V[f"cv-{k}-h"] = f(row["h_m"] * 1e3, 1)
            V[f"cv-{k}-nel"] = f(row["n_elements"], 0)
            V[f"cv-{k}-dofs"] = f(row["n_dofs"], 0)
            V[f"cv-{k}-top"] = f(row["top_deflection_m"] * 1e3, 4)
            V[f"cv-{k}-vm"] = f(row["max_von_mises_Pa"] / 1e6, 1)
            V[f"cv-{k}-vmaway"] = f(row["max_von_mises_away_from_corners_Pa"] / 1e6, 1)
            V[f"cv-{k}-f1"] = f(row["f1_Hz"], 2)
            for b, d in row["von_mises_per_body"].items():
                V[f"cv-{k}-{b}-max"] = f(d["max_Pa"] / 1e6, 1)
                V[f"cv-{k}-{b}-away"] = f(d["away_Pa"] / 1e6, 1)
        for k, v in cv["change_medium_to_fine"].items():
            V[f"cv-chg-{k.replace('_', '')}"] = f"{v:+.2f}"
        for b, d in cv["change_medium_to_fine_per_body"].items():
            V[f"cv-chg-{b}-max"] = f"{d['max_pct']:+.1f}"
            V[f"cv-chg-{b}-away"] = f"{d['away_pct']:+.1f}"
    V["alpha-cu"] = f(mt["copper"]["alpha"] * 1e6, 0)
    V["alpha-ep"] = f(mt["epoxy"]["alpha"] * 1e6, 0)
    sec = hc["sections"]
    V["EA-ratio"] = f(mt["epoxy"]["E"] * sec["A_k"] / (mt["copper"]["E"] * sec["A_c"]), 1)
    V["run-time-min"] = f(r.get("run_time_s", 0) / 60, 0)
    V["date"] = dt.date.today().strftime("%B %Y")
    return V


def tex_escape(s):
    return s.replace("%", r"\%").replace("_", r"\_").replace("&", r"\&")


def write_tex(r, V):
    GEN.mkdir(parents=True, exist_ok=True)
    lines = ["% Generated by src/make_tables.py from results/results.json. Do not edit.",
             r"\makeatletter",
             r"\newcommand{\V}[1]{\ifcsname val@#1\endcsname\csname val@#1\endcsname\else\errmessage{Missing value #1}\fi}"]
    for k, v in V.items():
        lines.append(rf"\expandafter\def\csname val@{k}\endcsname{{{tex_escape(str(v))}}}")
    lines.append(r"\makeatother")
    (GEN / "values.tex").write_text("\n".join(lines) + "\n")

    # Verification table
    rows = []
    for i, row in enumerate(r["verification"], 1):
        rows.append(f"{tex_escape(row['check'])} & {row['unit'].replace('C', r'\si{\celsius}')} & "
                    f"{V[f'ver{i}-an']} & {V[f'ver{i}-fea']} & {V[f'ver{i}-diff']} \\\\")
    (GEN / "verification_table.tex").write_text("\n".join([
        r"\begin{tabular}{p{10.2cm}lrrr}", r"\toprule",
        r"Check & Unit & Analytical & FEA & Diff. (\%) \\", r"\midrule", *rows, r"\bottomrule", r"\end{tabular}"]) + "\n")

    # Materials table from the inputs
    mt = r["inputs"]["materials"]
    names = {"copper": "Copper (conductor)", "epoxy": "Epoxy (core)", "aluminium": "Aluminium (flange)"}
    mrows = []
    for k, n in names.items():
        p = mt[k]
        res = f"{p['resistivity']:.3g}" if "resistivity" in p else "n/a"
        mrows.append(f"{n} & {p['E'] / 1e9:.0f} & {p['nu']:.2f} & {p['rho']:.0f} & {p['k']:g} & "
                     f"{p['alpha'] * 1e6:.0f} & {res} \\\\")
    (GEN / "materials_table.tex").write_text("\n".join([
        r"\begin{tabular}{lrrrrrr}", r"\toprule",
        r"Material & $E$ (GPa) & $\nu$ & $\rho$ (kg/m$^3$) & $k$ (W/(m K)) & $\alpha$ ($10^{-6}$/K) & $\rho_e$ ($\Omega$ m) \\",
        r"\midrule", *mrows, r"\bottomrule", r"\end{tabular}"]) + "\n")

    # Static results
    srows = []
    for key, name in [("S1", "S1 cantilever"), ("S2", "S2 self-weight"), ("S3", "S3 static-equivalent seismic")]:
        srows.append(f"{name} & {V[f'{key}-top-mm']} & {V[f'{key}-umax-mm']} & "
                     + " & ".join(f"{V[f'{key}-{b}-max']} ({V[f'{key}-{b}-away']})" for b in ["conductor", "core", "flange"])
                     + r" \\")
    (GEN / "static_table.tex").write_text("\n".join([
        r"\begin{tabular}{lrrrrr}", r"\toprule",
        r"Load case & Top defl. (mm) & Max $|u|$ (mm) & Conductor (MPa) & Core (MPa) & Flange (MPa) \\",
        r"\midrule", *srows, r"\bottomrule", r"\end{tabular}"]) + "\n")

    # Modal
    mrows = []
    for i, mode in enumerate(r["modal"]["modes"], 1):
        mrows.append(f"{i} & {V[f'f{i}']} & {tex_escape(mode['type'])} & {tex_escape(mode['dominant_region'])} & "
                     f"{V[f'f{i}-effx']} & {V[f'f{i}-effy']} \\\\")
    (GEN / "modal_table.tex").write_text("\n".join([
        r"\begin{tabular}{rrllrr}", r"\toprule",
        r"Mode & $f$ (Hz) & Type & Region & $m_{\mathrm{eff},x}$ (\%) & $m_{\mathrm{eff},y}$ (\%) \\",
        r"\midrule", *mrows, r"\bottomrule", r"\end{tabular}"]) + "\n")

    # Thermal-structural
    trows = []
    for key, name in [("TH", "Thermal, fixed flange face"), ("THSL", "Thermal, sliding flange face"),
                      ("THS1", "Thermal + S1, fixed flange face")]:
        trows.append(f"{name} & " + " & ".join(f"{V[f'{key}-{b}-max']} ({V[f'{key}-{b}-away']})"
                                               for b in ["conductor", "core", "flange"]) + r" \\")
    (GEN / "thermo_mech_table.tex").write_text("\n".join([
        r"\begin{tabular}{lrrr}", r"\toprule",
        r"Case & Conductor (MPa) & Core (MPa) & Flange (MPa) \\", r"\midrule", *trows,
        r"\bottomrule", r"\end{tabular}"]) + "\n")
    irows = []
    for iface, lab in [("flange_to_core", "Flange to core"), ("conductor_to_core", "Conductor to core")]:
        for b in r["thermo_mech"]["thermal_only"]["interfaces"][iface]:
            irows.append(f"{lab} & {b} side & {V[f'TH-{iface}-{b}-max']} ({V[f'TH-{iface}-{b}-away']}) & "
                         f"{V[f'THSL-{iface}-{b}-max']} ({V[f'THSL-{iface}-{b}-away']}) \\\\")
    (GEN / "interface_table.tex").write_text("\n".join([
        r"\begin{tabular}{llrr}", r"\toprule",
        r"Interface & Side & Fixed face (MPa) & Sliding face (MPa) \\", r"\midrule", *irows,
        r"\bottomrule", r"\end{tabular}"]) + "\n")

    # Convergence
    if "convergence" in r:
        crows = []
        for k in ["coarse", "medium", "fine"]:
            crows.append(f"{k.capitalize()} & {V[f'cv-{k}-h']} & {V[f'cv-{k}-nel']} & {V[f'cv-{k}-dofs']} & "
                         f"{V[f'cv-{k}-top']} & {V[f'cv-{k}-vm']} & {V[f'cv-{k}-vmaway']} & {V[f'cv-{k}-f1']} \\\\")
        crows.append(r"\midrule")
        crows.append(f"Change medium to fine (\\%) & & & & {V['cv-chg-topdeflectionpct']} & {V['cv-chg-maxvonmisespct']} & "
                     f"{V['cv-chg-maxvonmisesawayfromcornerspct']} & {V['cv-chg-f1pct']} \\\\")
        (GEN / "convergence_table.tex").write_text("\n".join([
            r"\begin{tabular}{lrrrrrrr}", r"\toprule",
            r"Mesh & $h$ (mm) & Elements & DOFs & Top defl. (mm) & Max VM (MPa) & VM away (MPa) & $f_1$ (Hz) \\",
            r"\midrule", *crows, r"\bottomrule", r"\end{tabular}"]) + "\n")


def md_verification_table(r, V):
    out = ["| Check | Analytical | FEA | Difference |", "|---|---:|---:|---:|"]
    for i, row in enumerate(r["verification"], 1):
        u = row["unit"]
        out.append(f"| {row['check']} | {V[f'ver{i}-an']} {u} | {V[f'ver{i}-fea']} {u} | {V[f'ver{i}-diff']} % |")
    return "\n".join(out)


def fill_templates(r, V):
    V = dict(V)
    V["verification-table-md"] = md_verification_table(r, V)
    targets = {"README.md": ROOT / "README.md", "EXPLAINER.md": ROOT / "docs" / "EXPLAINER.md",
               "CV_ENTRY.md": ROOT / "docs" / "CV_ENTRY.md"}
    for name, dest in targets.items():
        src = TEMPLATES / name
        if not src.exists():
            continue
        text = src.read_text()
        missing = sorted(set(k for k in re.findall(r"\{\{([^}]+)\}\}", text) if k not in V))
        if missing:
            raise KeyError(f"{name}: no value for {missing}")
        text = re.sub(r"\{\{([^}]+)\}\}", lambda m: str(V[m.group(1)]), text)
        header = f"<!-- Generated by src/make_tables.py from docs/templates/{name} and results/results.json. -->\n"
        dest.write_text(header + text)
        print("written", dest.relative_to(ROOT))


def main():
    r = read_results()
    V = values(r)
    write_tex(r, V)
    fill_templates(r, V)
    print(f"{len(V)} values written to report/generated/values.tex")


if __name__ == "__main__":
    main()

### `run_all.py`: Runs everything

In [ ]:
%%writefile src/run_all.py
"""Run the whole study: mesh, static, modal, thermal, thermal-structural, hand calculations,
verification, mesh convergence, figures, report tables and documents.

Usage:  python src/run_all.py                 (everything)
        python src/run_all.py --skip-convergence   (keeps the convergence results already stored)
"""
import argparse
import gc
import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import dolfinx  # noqa: E402

from common import RESULTS_DIR, RESULTS_FILE, load_config, update_results  # noqa: E402


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--skip-convergence", action="store_true")
    args = ap.parse_args()
    t0 = time.time()
    cfg = load_config()
    RESULTS_DIR.mkdir(exist_ok=True)
    old = json.loads(RESULTS_FILE.read_text()) if RESULTS_FILE.exists() else {}
    RESULTS_FILE.write_text("{}")
    if args.skip_convergence and "convergence" in old:
        update_results("convergence", old["convergence"])

    import gmsh
    import numpy as np
    from petsc4py import PETSc
    update_results("software", {"dolfinx": dolfinx.__version__, "gmsh": gmsh.__version__,
                                "petsc": ".".join(map(str, PETSc.Sys.getVersion())), "numpy": np.__version__})
    update_results("inputs", cfg)

    import geometry, hand_calcs, modal, static, thermal, thermo_mech
    from common import derived_geometry

    print("== Stage 2: geometry and mesh", flush=True)
    data, stats = geometry.create_mesh(cfg)
    update_results("mesh", stats)
    update_results("geometry", {k: float(v) for k, v in derived_geometry(cfg).items()})
    print(stats, flush=True)

    print("== Hand calculations", flush=True)
    hand_calcs.run(cfg)

    print("== Stage 3: static", flush=True)
    model, us, _ = static.run(data, cfg)

    print("== Stage 4: modal", flush=True)
    modal.run(data, cfg, model=model)

    print("== Stage 5: thermal", flush=True)
    T, _ = thermal.run(data, cfg)

    print("== Stage 6: thermal-structural", flush=True)
    thermo_mech.run(data, cfg, T, model=model, u_s1=us["S1_cantilever"])

    print("== Verification", flush=True)
    for row in hand_calcs.verify():
        print(f"  {row['check']:85s} analytical {row['analytical']:9.4f}  FEA {row['fea']:9.4f} {row['unit']:3s} "
              f"diff {row['difference_pct']:+7.2f} %")

    model.ksp.destroy()
    del model, us, T, data
    gc.collect()

    if not args.skip_convergence:
        print("== Stage 7: mesh convergence", flush=True)
        import convergence
        convergence.run(cfg)

    print("== Stage 8: figures", flush=True)
    import make_figures
    make_figures.main()

    print("== Stage 9/10: report tables and documents", flush=True)
    update_results("run_time_s", time.time() - t0)
    import make_tables
    make_tables.main()
    print(f"done in {time.time() - t0:.0f} s")


if __name__ == "__main__":
    main()

## 5. Templates for the generated documents and the LaTeX report
These contain `{{placeholders}}` (Markdown) and `\V{key}` macros (LaTeX) that are filled from `results.json`.

In [ ]:
%%writefile docs/templates/README.md
# Transformer Bushing FEA: Static, Modal and Thermal-Structural

A self-directed, simplified finite element study of a dry-type (resin-impregnated) transformer bushing, built only with open-source Python tools (Gmsh, FEniCSx/dolfinx, SLEPc, PyVista, Matplotlib). The model has three bonded bodies (copper conductor tube, epoxy core, aluminium flange) and covers a static terminal (cantilever) load, self-weight, a static-equivalent horizontal acceleration, modal analysis, steady-state Joule heating and the resulting thermal stress. Every FEA result is checked against a hand calculation. The load cases are *based on* the type of checks in IEC 60137 and IEEE 693, but the study does **not** claim compliance with these standards, qualification or a rated design. All material values are typical literature values, assumed.

![Geometry and mesh](figures/fig1_geometry_mesh.png)

## Key results (mesh h = {{mesh-h-mm}} mm, {{mesh-nel}} quadratic tetrahedra, {{ndofs}} displacement DOFs)

| Quantity | Value |
|---|---:|
| S1 top deflection under F = {{F-N}} N | {{S1-top-mm}} mm |
| First natural frequency (bending, two near-equal directions) | {{f1}} Hz and {{f2}} Hz |
| Hot spot at I = {{I-A}} A, ambient {{T-amb}} C | {{Tmax}} C |
| Thermal von Mises stress, max in conductor / core / flange (fixed flange face) | {{TH-conductor-max}} / {{TH-core-max}} / {{TH-flange-max}} MPa |

## Verification against hand calculations

{{verification-table-md}}

The two-segment beam and 1D conduction checks are the meaningful ones; the uniform-beam and free-copper rows are kept to show why the simpler formulas do not fit this geometry. See `report/report.pdf` for the explanations.

## How to run

```bash
conda env create -f environment.yml      # or: micromamba create -f environment.yml
conda activate bushing-fea
python src/run_all.py                    # all stages ({{run-time-min}} minutes on the test machine)
python src/run_all.py --skip-convergence # keeps the stored convergence results, much faster
cd report && pdflatex report.tex && pdflatex report.tex
```

`run_all.py` regenerates `results/results.json`, every figure in `figures/`, the tables in `report/generated/` and the numbers in this README, `docs/EXPLAINER.md` and `docs/CV_ENTRY.md` (from `docs/templates/`). All parameters are in `config.yaml`. Tested with dolfinx {{ver-dolfinx}}, Gmsh {{ver-gmsh}}, PETSc {{ver-petsc}}. The fine mesh of the convergence study needs a lot of memory; reduce `convergence_h` in `config.yaml` on a smaller machine.

## Repository layout

```
config.yaml      all geometry, material, load and mesh parameters (SI units)
src/             geometry.py, static.py, modal.py, thermal.py, thermo_mech.py, hand_calcs.py,
                 convergence.py, make_figures.py, make_tables.py, run_all.py
results/         results.json and field files (.vtu, open in ParaView)
figures/         PNG figures, 300 dpi
report/          report.tex (simulation report) and generated tables
docs/            EXPLAINER.md (plain-language walkthrough), CV_ENTRY.md
```

## Limitations

- Simplified geometry: no silicone sheds or housing, no condenser foils, no oil-side details, no draw lead in the bore.
- Perfect bonding between bodies, linear elastic isotropic materials, properties at a single temperature.
- The flange bottom face is fully fixed. This idealised mount over-predicts thermal stress near the flange; a sliding-support sensitivity case is included to show how much.
- Thermal stress assumes a stress-free state at {{T-ref}} C. A real resin-impregnated core is cured at elevated temperature and has residual stress, so the thermal stress values are indicative only.
- Static-equivalent acceleration only; no response spectrum or time-history seismic analysis to IEEE 693.
- Resistivity at 20 C (no temperature dependence), no short-circuit electromagnetic forces, no dielectric losses.

In [ ]:
%%writefile docs/templates/EXPLAINER.md
# Explainer: the bushing FEA project in plain language

This note is for interview preparation. It explains what each part of the study does, why a bushing designer would care, how the inputs were chosen, what the checks show and where the model is weak. All numbers are filled in from `results/results.json`.

## 1. What is modelled

A transformer bushing carries the high-voltage conductor through the grounded tank cover. In a resin-impregnated (dry) bushing, the conductor is surrounded by an epoxy core, and a metal flange bolts the bushing to the tank. The model keeps only these three parts:

- copper conductor tube, {{Dco-mm}} mm outer and {{Dci-mm}} mm inner diameter, {{Lcond-m}} m long, with an empty bore,
- epoxy core, {{Dk-mm}} mm diameter, {{Lcore-m}} m long, bonded to the conductor,
- aluminium flange ring, {{Df-mm}} mm outer diameter, {{tf-mm}} mm thick, bonded to the core with its bottom face {{zf0-m}} m above the core bottom.

Above the flange the bushing is a cantilever {{L-beam-m}} m long: {{a-beam-m}} m of core with conductor, then {{b-beam-m}} m of bare conductor (the terminal stud). Below the flange hangs a shorter part that would sit in the transformer.

Left out on purpose: sheds and housing, condenser foils, oil-side details, the draw lead in the bore, bolts and the tank. All contacts are perfectly bonded, and all materials are linear elastic and isotropic with typical literature values (assumed, not from a datasheet).

The mesh has {{mesh-nel}} quadratic (10-node) tetrahedra with a global size of {{mesh-h-mm}} mm, refined to {{mesh-hc-mm}} mm at the conductor and {{mesh-hf-mm}} mm at the flange. The worst element has a scaled Jacobian of {{mesh-qmin}} (mean {{mesh-qmean}}), which is acceptable.

## 2. The analyses and why a designer cares

**Static, S1 cantilever.** A line connected to the terminal pulls sideways on the bushing (conductor tension, wind, ice). IEC 60137 has a cantilever withstand test for this. I applied an illustrative {{F-N}} N horizontally on the top end face; the real test load depends on the bushing rating and must be taken from the standard. Result: top deflection {{S1-top-mm}} mm. The highest stress is in the copper where it leaves the core top ({{S1-conductor-max}} MPa at the corner, {{S1-conductor-away}} MPa at {{excl-mm}} mm from it), because the bare stud carries the full bending moment on a small section. The flange sees only {{S1-flange-away}} MPa away from corners.

**Static, S2 self-weight.** A sanity case. The bushing model weighs {{mass-kg}} kg, and the largest displacement is only {{S2-umax-um}} micrometres, so self-weight matters little for this short, stiff model.

**Static, S3 static-equivalent seismic.** A horizontal acceleration of {{seis-g}} g on all mass, plus gravity. The top moves {{S3-top-um}} micrometres. This is a quick screening check, not a seismic qualification. IEEE 693 uses response spectra or shake-table tests, and the dynamic amplification depends on the natural frequency (next point).

**Modal.** Earthquake ground motion has most of its energy at low frequencies, so a bushing whose first natural frequency is well above that responds almost as a rigid body, while a low frequency means the ground motion is amplified. The first two modes are bending in two perpendicular directions at {{f1}} Hz and {{f2}} Hz. For an axisymmetric body they should be equal; the {{f12-diff-pct}} % difference comes only from the unstructured mesh, so this is a useful symmetry check. Modes 3 and 4 ({{f3}} Hz) are bending of the part below the flange, and modes 5 and 6 ({{f5}} Hz) are second bending of the upper part. In a real bushing the long porcelain or composite housing, the oil and a flexible tank cover lower these frequencies a lot.

**Thermal.** Load current heats the conductor (I squared R). The heat must cross the epoxy, which conducts poorly, and leave by natural convection. The insulation temperature limits the bushing rating (IEC 60137 sets temperature-rise limits for the hot spot). At {{I-A}} A the conductor generates {{Qp-Wm}} W per metre, {{Qtot-W}} W in total. With convection h = {{h-conv}} W/(m2 K) to {{T-amb}} C air, the hot spot is {{Tmax}} C, at the top end of the conductor. This is because I treated the exposed conductor ends as insulated, so their heat has to flow along the copper into the core. In a real bushing the terminal connection would carry heat away (or bring heat in from a hot line), so the hot spot location depends strongly on that boundary condition.

**Thermal-structural.** Copper expands about half as much as epoxy per kelvin ({{alpha-cu}} versus {{alpha-ep}} micrometres per metre per kelvin). When the bonded bushing warms up, the two materials fight each other and stress builds up at the interfaces. With a stress-free state at {{T-ref}} C and the flange face fully fixed, the maximum von Mises stress is {{TH-conductor-max}} MPa in the conductor, {{TH-core-max}} MPa in the core and {{TH-flange-max}} MPa in the flange, all at the flange level. These numbers are much larger than the {{F-N}} N cantilever stresses, and adding S1 changes them very little.

I checked how much of this comes from the idealised fixed support by repeating the case with a sliding support (axial restraint only, free radial growth). The maxima drop to {{THSL-conductor-max}}, {{THSL-core-max}} and {{THSL-flange-max}} MPa. So a large part of the fixed-support result is caused by the rigid mount, and a real assessment needs the tank cover and bolt stiffness. A second strong effect is the stress-free temperature: a real resin core is cured hot on the conductor, so its residual stress state is not zero at {{T-ref}} C.

## 3. How the boundary conditions and loads were chosen

| Item | Choice in the model | What a real design would do |
|---|---|---|
| Support | Flange bottom face fully fixed | Bolted joint on a flexible tank cover, possibly with contact and bolt preload |
| Terminal load | {{F-N}} N uniform traction on the top end face, illustrative | Rated cantilever test load from IEC 60137, applied at the terminal point |
| Seismic | {{seis-g}} g static horizontal plus gravity | Response spectrum analysis to IEEE 693, with the tank flexibility (amplification factors) |
| Heat source | I squared R with resistivity at 20 C | Temperature-dependent resistivity, skin effect, dielectric losses, oil and tank temperatures |
| Cooling | h = {{h-conv}} W/(m2 K) to {{T-amb}} C on core and flange | Correlations for natural convection and radiation on the real housing, oil-side heat transfer |
| Thermal stress | Stress-free at {{T-ref}} C, bonded | Cure temperature and residual stress of the resin, temperature-dependent epoxy modulus |

## 4. What the verification shows

{{verification-table-md}}

**Static deflection.** The uniform composite beam ({{ver1-an}} mm) is not a fair check, because it gives the bare {{b-beam-m}} m stud the bending stiffness of the whole core. The two-segment beam ({{ver2-an}} mm) is the right check, and the FEA is {{ver2-diff}} % more flexible. About {{share-overhang-pct}} % of the beam-theory deflection comes from the stud (its own bending plus the rotation of the core top). The beam model clamps the core rigidly at the flange and assumes that the conductor is rigidly built in at the core top; in 3D the epoxy around the conductor and at the flange deforms locally, which adds flexibility. Shear deformation of the core, which beam theory ignores, adds a little more. Figure 8 compares the deflection lines.

**First natural frequency.** The uniform formula gives {{f1-uniform}} Hz against {{f1}} Hz from FEA. The error is large because the formula assumes the composite mass ({{meff}} kg/m) all the way to the tip, while the stud only has {{mc}} kg/m, and tip mass controls the first bending frequency. A 1D beam model with the two real segments gives {{f1-twoseg}} Hz, {{ver4-diff}} % from FEA. The FEA is slightly higher, probably because the flange stiffens the core over its {{tf-mm}} mm height, so the effective clamp is a little higher than the flange bottom face.

**Radial temperature.** At core mid-height the 1D long-cylinder solution with the local generation ({{Qp-Wm}} W/m) predicts {{mid-Tco-an}} C at the conductor surface; the FEA gives {{mid-Tco-fea}} C. The FEA is higher because the heat generated in the insulated conductor ends travels along the copper into the core section, so the radial heat flow at mid-height is {{mid-Qlocal}} W/m ({{mid-Qlocal-excess-pct}} % more than generated locally). If the 1D formula uses that local heat flow, the temperature drop across the epoxy matches the FEA within {{ver8-absdiff}} %. So the conduction physics and the mesh are right; the difference is a real 3D effect of a short bushing with insulated ends. This is the same kind of check as the analytical comparison in my thesis on multilayer cable insulation. The heat balance also closes: {{Q-gen}} W generated and {{Q-out}} W convected.

**Thermal elongation.** Free copper expansion over the {{L-beam-m}} m above the flange gives {{el-free-mm}} mm, but the FEA top displacement is {{el-fea-mm}} mm. The epoxy is {{EA-ratio}} times stiffer than the copper in the axial direction (E times A) and expands twice as much, so the bonded core pulls the conductor with it. An EA-weighted composite estimate gives {{el-comp-mm}} mm, within {{ver10-absdiff}} % of the FEA.

## 5. Mesh convergence

| Mesh | h (mm) | Elements | Top deflection (mm) | Max stress (MPa) | Max stress away from corners (MPa) | f1 (Hz) |
|---|---:|---:|---:|---:|---:|---:|
| Coarse | {{cv-coarse-h}} | {{cv-coarse-nel}} | {{cv-coarse-top}} | {{cv-coarse-vm}} | {{cv-coarse-vmaway}} | {{cv-coarse-f1}} |
| Medium | {{cv-medium-h}} | {{cv-medium-nel}} | {{cv-medium-top}} | {{cv-medium-vm}} | {{cv-medium-vmaway}} | {{cv-medium-f1}} |
| Fine | {{cv-fine-h}} | {{cv-fine-nel}} | {{cv-fine-top}} | {{cv-fine-vm}} | {{cv-fine-vmaway}} | {{cv-fine-f1}} |

From medium to fine, the top deflection changes by {{cv-chg-topdeflectionpct}} % and f1 by {{cv-chg-f1pct}} %, so these are converged.

The stresses are a different story, and it is worth explaining honestly:

- The **peak** stresses rise with every refinement (epoxy {{cv-coarse-core-max}}, {{cv-medium-core-max}}, {{cv-fine-core-max}} MPa; flange {{cv-coarse-flange-max}}, {{cv-medium-flange-max}}, {{cv-fine-flange-max}} MPa). They sit at sharp re-entrant or bi-material corners, where linear elasticity gives an infinite stress, so a finer mesh only shows a higher number.
- The maxima at **{{excl-mm}} mm from those corners** do not keep rising. In the copper they are converged ({{cv-chg-conductor-away}} %). In the epoxy and flange they still move by {{cv-chg-core-away}} % and {{cv-chg-flange-away}} % (epoxy {{cv-coarse-core-away}}, {{cv-medium-core-away}}, {{cv-fine-core-away}} MPa), because {{excl-mm}} mm is only one or two elements near the flange and the point sits in a steep gradient.

So the global quantities are converged, the copper stress is converged, and the local stresses at the flange are not. To get a usable local stress I would model the real fillet radius with a local mesh refinement, or use a linearised or hot-spot stress method.

## 6. Ten likely interview questions

**1. Why FEniCSx and not Ansys?**
I work on a MacBook, where Ansys does not run, and I wanted a project anyone can rerun from GitHub. FEniCSx is a research-grade open-source FE library; writing the weak forms myself made me check every boundary condition and material assignment. The physics and verification steps are the same as I used with Ansys in my thesis, and I would expect to switch back to Ansys Mechanical at work.

**2. Why does the stress not converge at the corner?**
At a sharp re-entrant corner or where two materials meet at an edge, the elastic solution has a singularity: stress grows without bound as you approach the point. Each mesh refinement samples closer to it, so the peak keeps rising. I report the peak and the maximum at {{excl-mm}} mm from the singular edges. The second value does not keep growing, but in the epoxy it still moves by {{cv-chg-core-away}} % between my medium and fine meshes, so I would not call it converged either. In a real design I would model the actual fillet radius or use a hot-spot or linearised stress method.

**3. Your deflection is {{ver2-absdiff}} % off the beam formula. Is the model wrong?**
No. The beam model assumes a perfect clamp at the flange and a rigid joint where the stud leaves the core. Figure 8 shows the deflection line, so I can see where the extra flexibility enters. A few percent extra flexibility from local deformation is expected for a composite with a soft core. The mesh study shows the FEA deflection is converged to {{cv-chg-topdeflectionpct}} %.

**4. How would you do a proper seismic analysis?**
Follow IEEE 693: get the bushing natural frequencies with the real housing and the flexible tank cover (often with an amplification factor for the tank), then run a response spectrum analysis with the required spectrum and damping, combine modes (SRSS or CQC) and directions, and compare the stresses at the flange, the porcelain or composite housing and the gaskets with allowables. For high qualification levels it ends in a shake-table test. The effective modal masses I computed ({{f1-effx}} % in mode 1) are the input to such a modal combination.

**5. What would short-circuit forces add?**
During a short circuit, current many times the rated value flows for a short time. The magnetic field from neighbouring phases gives dynamic electromagnetic forces on the conductor and terminal, and the extra I squared R heats the conductor quickly (adiabatically). Both would be new load cases: a transient structural case with the force time history (or an equivalent static load from IEC 60865-type calculations) and a short transient thermal case.

**6. Why is the hot spot at the top end and not in the middle?**
Because I made the exposed conductor ends insulated, so heat generated there can only escape by flowing along the copper into the core. In a real installation the terminal connection is a heat path, either out or in, so the hot spot location is a modelling choice I would discuss with the thermal team.

**7. The thermal stresses are high. Is the bushing failing?**
I would not conclude that. The values depend strongly on two assumptions: the fully fixed flange face (the sliding support roughly halves the peaks) and a stress-free state at {{T-ref}} C. The real cure temperature, residual stress, the temperature dependence of epoxy properties and the real mounting stiffness all need to be included before comparing with allowables. What the study does show is where the stress concentrates (flange level and conductor exit) and that thermal mismatch dominates over the cantilever load in this model.

**8. Why can you add the thermal and S1 stresses?**
The model is linear (small strains, linear elastic, bonded contacts, no friction). For a linear model the displacement fields superpose. I added the displacement fields and recalculated the von Mises stress, because von Mises itself is not additive.

**9. How did you check the modal results?**
Three ways: the two bending modes have almost equal frequencies ({{f12-diff-pct}} % apart) as symmetry requires; the effective masses show each mode's direction; and a 1D two-segment beam model gives {{f1-twoseg}} Hz against {{f1}} Hz. f1 also changes by only {{cv-chg-f1pct}} % between the medium and fine meshes.

**10. What would you add next?**
The real shed and housing geometry, the oil side and tank cover stiffness, contact instead of bonding at the flange, temperature-dependent resistivity and material properties, a response spectrum seismic case and a short-circuit force case. I would also compare against test data if available, because verification against hand calculations only shows that the model solves the equations correctly, not that the equations represent the real bushing.

In [ ]:
%%writefile docs/templates/CV_ENTRY.md
# CV entry

Every number below is filled from `results/results.json` by `src/make_tables.py`.

```latex
\projline{Transformer Bushing FEA: Static, Modal and Thermal-Structural}{FEniCSx, Gmsh, Python, self-directed, {{date}}}
\begin{itemize}
  \item Built an open-source 3D finite element model of a simplified transformer bushing in Python: static terminal load, modal analysis ($f_1$ = {{f1}} Hz) and Joule-heating thermal-structural coupling.
  \item Verified against beam theory (deflection within {{ver2-absdiff}}\,\%, $f_1$ within {{ver4-absdiff}}\,\%) and analytical radial heat conduction (within {{ver8-absdiff}}\,\%); mesh convergence study; github.com/SirTNT/bushing-fea.
\end{itemize}
```

## Skills section, one line

```latex
Open-source FEA in Python (FEniCSx/dolfinx, Gmsh, SLEPc): static, modal and coupled thermal-structural analysis with analytical verification.
```

## Notes for using these lines

- "Analytical radial heat conduction (within {{ver8-absdiff}} %)" refers to the temperature drop across the epoxy at core mid-height, with the 1D formula fed the local radial heat flow taken from the FEA. With the locally generated heat instead, the difference is {{ver7-absdiff}} %, because heat from the insulated conductor ends flows along the copper into the core region. Be ready to explain both numbers.
- "Deflection within {{ver2-absdiff}} %" is against the two-segment beam (core plus conductor, then bare conductor). The uniform composite beam differs by {{ver1-absdiff}} % and is not a fair comparison for this geometry.
- "$f_1$ within {{ver4-absdiff}} %" is against a 1D two-segment beam model. The uniform cantilever formula with composite properties differs by {{ver3-absdiff}} %, because it places the full composite mass on the bare conductor at the tip.

In [ ]:
%%writefile report/report.tex
\documentclass[11pt,a4paper]{article}
\usepackage[margin=1.8cm]{geometry}
\usepackage{amsmath,amssymb}
\usepackage{adjustbox}
\usepackage{graphicx}
\usepackage{booktabs}
\usepackage{siunitx}
\usepackage{float}
\usepackage[font=small,labelfont=bf]{caption}
\usepackage[hidelinks]{hyperref}
\usepackage{enumitem}
\setlist{nosep,leftmargin=*}
\setlength{\parskip}{3pt}
\setlength{\parindent}{0pt}
\graphicspath{{../figures/}}
\renewcommand{\topfraction}{0.9}
\renewcommand{\bottomfraction}{0.8}
\renewcommand{\textfraction}{0.07}
\renewcommand{\floatpagefraction}{0.85}
\setlength{\textfloatsep}{8pt plus 2pt}
\setlength{\floatsep}{6pt plus 2pt}
\input{generated/values.tex}

\title{Simplified Transformer Bushing: Static, Modal and Thermal-Structural Finite Element Study}
\author{Archis Mahore}
\date{\V{date}}

\begin{document}
\maketitle
\vspace{-1.5em}

\section{Purpose and scope}
This report describes a self-directed finite element study of a simplified dry-type (resin-impregnated) condenser bushing. The aim is to show a complete simulation workflow: geometry and mesh, static, modal, thermal and thermal-structural analyses, verification against hand calculations and a mesh convergence study, all with open-source tools (Gmsh~\V{ver-gmsh}, FEniCSx/dolfinx~\V{ver-dolfinx}, PETSc/SLEPc~\V{ver-petsc}). The load cases are based on the type of checks in IEC~60137 (bushings) and IEEE~693 (seismic). The study is \emph{not} a design, rating or qualification of a bushing, does not claim compliance with these standards, and uses assumed material data. Every number in this report is read from \texttt{results/\allowbreak results.json}, which is produced by \texttt{src/\allowbreak run\_all.py}.

\section{Model}
\textbf{Geometry.} Three coaxial bodies along the vertical $z$ axis ($z=0$ at the core bottom): a copper conductor tube ($\varnothing$\V{Dco-mm}/\V{Dci-mm}~mm, length \V{Lcond-m}~m, $z=\V{zc0-m}$ to \V{zc1-m}~m), an epoxy core bonded around it ($\varnothing$\V{Dk-mm}~mm, length \V{Lcore-m}~m) and an aluminium flange ring ($\varnothing$\V{Df-mm}~mm, \V{tf-mm}~mm thick, bottom face at $z=\V{zf0-m}$~m). The bore of the conductor is empty. Simplifications: no silicone sheds or housing, no condenser foils, no oil-side details, perfect bonding between bodies, linear elastic isotropic materials with properties at one temperature.

\textbf{Materials} (Table~\ref{tab:mat}): typical literature values, assumed; not taken from a datasheet.
\begin{table}[!htb]\centering\small
\caption{Material properties (assumed). $\rho_e$ is the electrical resistivity at \SI{20}{\celsius}.}\label{tab:mat}
\begin{adjustbox}{max width=\textwidth}\input{generated/materials_table.tex}\end{adjustbox}
\end{table}

\textbf{Mesh.} The bodies are fragmented in Gmsh (OpenCASCADE) so the mesh is conforming at the bonded interfaces. Quadratic 10-node tetrahedra, global size $h=\V{mesh-h-mm}$~mm (the single mesh parameter), refined to \V{mesh-hc-mm}~mm at the conductor and \V{mesh-hf-mm}~mm at the flange: \V{mesh-nel} elements, \V{mesh-nnodes} nodes, \V{ndofs} displacement degrees of freedom. Minimum scaled Jacobian \V{mesh-qmin} (mean \V{mesh-qmean}). Displacement and temperature use quadratic Lagrange elements.

\textbf{Boundary conditions and load cases.} The flange bottom face is fully fixed in all structural cases (a sliding support is used once as a sensitivity case, Section~\ref{sec:tm}).
\begin{itemize}
\item S1 cantilever: horizontal force $F=\V{F-N}$~N as a uniform traction on the top conductor end face. Illustrative value; the IEC~60137 test load depends on the rating.
\item S2 self-weight: gravity $g=\V{g}$~m/s$^2$.
\item S3 static-equivalent seismic: horizontal acceleration \V{seis-g}\,$g$ on all mass plus gravity. A simplified screening check, not a seismic qualification.
\item Modal: same support, first \V{n-modes} modes, SLEPc Krylov-Schur with shift-invert about zero.
\item Thermal: Joule heating $q=I^2\rho_e/A^2$ in the copper with $I=\V{I-A}$~A ($q=\V{q-Wm3}$~W/m$^3$, \V{Qp-Wm}~W/m, \V{Qtot-W}~W in total), resistivity at \SI{20}{\celsius}; convection $h=\V{h-conv}$~W/(m$^2$\,K) to \V{T-amb}~\si{\celsius} air on the outer surfaces of core and flange; bore and exposed conductor surfaces insulated (conservative).
\item Thermal-structural: the temperature field as thermal strain $\alpha(T-T_\mathrm{ref})$, stress-free at $T_\mathrm{ref}=\V{T-ref}$~\si{\celsius}, no mechanical load; then combined with S1 by superposition of displacements (the model is linear).
\end{itemize}
Stress at re-entrant and bi-material corners is singular. Each maximum von Mises stress is therefore given twice: the peak, and the maximum at more than \V{excl-mm}~mm from the singular ring edges (flange inner edges, edges of the fixed face, conductor exits from the core).

\begin{figure}[!htb]\centering
\includegraphics[width=0.40\textwidth]{fig1_geometry_mesh.png}
\caption{Geometry and mesh on the axial cut $y=0$.}\label{fig:geo}
\end{figure}

\section{Verification}
Table~\ref{tab:ver} compares each hand calculation with the FEA. The beam checks treat the part above the flange as a cantilever of length $L=\V{L-beam-m}$~m clamped at the flange bottom face: \V{a-beam-m}~m of core and conductor ($EI=\V{EIeff}$~N\,m$^2$), then \V{b-beam-m}~m of bare conductor ($EI=\V{EIc}$~N\,m$^2$).

\begin{table}[!htb]\centering\small
\caption{Verification of the FEA against analytical solutions. Difference $=(\text{FEA}-\text{analytical})/\text{analytical}$.}\label{tab:ver}
\begin{adjustbox}{max width=\textwidth}\input{generated/verification_table.tex}\end{adjustbox}
\end{table}

\begin{itemize}
\item \textbf{Deflection.} The uniform composite beam, $\delta=FL^3/(3EI_\mathrm{eff})$, gives the bare conductor the stiffness of the full section and so is too stiff. The two-segment beam is the fair check; the FEA is \V{ver2-diff}\,\% more flexible because the 3D model includes local deformation of the epoxy at the flange and around the conductor exit, which beam theory treats as rigid joints, plus a small shear deflection. Figure~\ref{fig:defl} shows the deflection lines.
\item \textbf{First frequency.} $f_1=(1.875^2/2\pi)\sqrt{EI/(mL^4)}$ with the composite mass per length (\V{meff}~kg/m) puts far too much mass at the tip, where the bare conductor has only \V{mc}~kg/m. A 1D Euler-Bernoulli model of the two segments (\V{n-beam-el} beam elements) agrees with the FEA within \V{ver4-absdiff}\,\%.
\item \textbf{Radial temperature.} The 1D long-cylinder solution (uniform generation in the tube, conduction through the epoxy, convection at the surface) under-predicts the rise at mid-height, because heat generated in the insulated conductor ends flows along the copper into the core region: the local radial heat flow there is \V{mid-Qlocal}~W/m, \V{mid-Qlocal-excess-pct}\,\% more than the local generation. With this local heat flow the same formula matches the FEA temperature drop across the epoxy within \V{ver8-absdiff}\,\%. The global energy balance closes (\V{Q-gen}~W generated, \V{Q-out}~W convected).
\item \textbf{Thermal elongation.} Free copper expansion $\alpha L\,\overline{\Delta T}$ ($\overline{\Delta T}=\V{el-dTcond}$~K) under-predicts the FEA top displacement, because the stiffer epoxy, which expands twice as much, pulls the conductor. An $EA$-weighted composite strain for the bonded segment plus free expansion of the bare conductor agrees within \V{ver10-absdiff}\,\%.
\end{itemize}

\section{Results}
\subsection{Static}
\begin{table}[!htb]\centering\small
\caption{Static results. Von Mises stress per body: peak (away from corners).}\label{tab:static}
\begin{adjustbox}{max width=\textwidth}\input{generated/static_table.tex}\end{adjustbox}
\end{table}
Under S1 the top deflects \V{S1-top-mm}~mm. The highest stress is in the copper where it leaves the core top (\V{S1-conductor-max}~MPa at the corner, \V{S1-conductor-away}~MPa away from it), close to the beam value $Fb\,r_o/I=\V{sig-cond-beam-MPa}$~MPa. Stress in the flange stays below \V{S1-flange-max}~MPa (Figure~\ref{fig:s1}). S2 and S3 give very small values for this short, stiff model (S3 top deflection \V{S3-top-um}~\si{\micro\metre}).

\begin{figure}[!htb]\centering
\includegraphics[height=5.0cm]{fig2_s1_displacement.png}\hfill
\includegraphics[height=5.0cm]{fig3_s1_von_mises_flange.png}
\caption{S1: displacement (left, deformation scaled) and von Mises stress with a zoom at the flange (right).}\label{fig:s1}
\end{figure}


\begin{figure}[!htb]\centering
\begin{minipage}[b]{0.50\textwidth}\centering
\includegraphics[width=\textwidth]{fig8_s1_deflection_profile.png}
\caption{S1 deflection along the conductor above the flange: FEA and two-segment beam.}\label{fig:defl}
\end{minipage}\hfill
\begin{minipage}[b]{0.36\textwidth}\centering
\includegraphics[width=\textwidth]{fig4_mode_shapes.png}
\caption{First two mode shapes on the cuts $y=0$ and $x=0$ (deformation scaled).}\label{fig:modes}
\end{minipage}
\end{figure}

\subsection{Modal}
\begin{table}[!htb]\centering\small
\caption{Natural frequencies and effective modal mass (fraction of the total mass, \V{mass-kg}~kg).}\label{tab:modal}
\begin{adjustbox}{max width=\textwidth}\input{generated/modal_table.tex}\end{adjustbox}
\end{table}
The first two modes are bending of the part above the flange in two perpendicular planes at \V{f1}~Hz and \V{f2}~Hz. Their difference of \V{f12-diff-pct}\,\% comes only from the unstructured mesh, which confirms the symmetry of the model. The degenerate pairs were rotated to align with $x$ and $y$ before plotting (Figure~\ref{fig:modes}).


\subsection{Thermal}
The hot spot is \V{Tmax}~\si{\celsius} (a rise of \V{Tmax-rise}~K over ambient) at the top end of the conductor ($z=\V{Tmax-z}$~m), because the exposed conductor ends are insulated in this model and their heat must reach the core by conduction along the copper. The maximum epoxy temperature is \V{Tmax-core}~\si{\celsius} and the flange stays at \V{Tmax-flange}~\si{\celsius}. At core mid-height the FEA gives \V{mid-Tco-fea}~\si{\celsius} at the conductor surface and \V{mid-Tk-fea}~\si{\celsius} at the core surface (Figure~\ref{fig:temp}).

\begin{figure}[!htb]\centering
\includegraphics[width=0.62\textwidth]{fig5_temperature.png}
\caption{Temperature on the axial cut and radial profile at core mid-height, FEA and 1D analytical.}\label{fig:temp}
\end{figure}

\subsection{Thermal-structural}\label{sec:tm}
\begin{table}[!htb]\centering\footnotesize
\begin{minipage}[t]{0.49\textwidth}\centering
\caption{Thermal-structural von Mises stress per body: peak (away from corners).}\label{tab:tm}
\begin{adjustbox}{max width=\textwidth}\input{generated/thermo_mech_table.tex}\end{adjustbox}
\end{minipage}\hfill
\begin{minipage}[t]{0.49\textwidth}\centering
\caption{Thermal-only von Mises stress on each side of the bonded interfaces: peak (away from corners).}\label{tab:if}
\begin{adjustbox}{max width=\textwidth}\input{generated/interface_table.tex}\end{adjustbox}
\end{minipage}
\end{table}
The thermal stresses are much larger than those from S1, and adding S1 changes the maxima very little (Table~\ref{tab:tm}). They are driven by the expansion mismatch between copper (\V{alpha-cu}$\times10^{-6}$/K) and epoxy (\V{alpha-ep}$\times10^{-6}$/K) and by the fully fixed flange face, which prevents the radial growth of the flange and so clamps the core and conductor at the flange level, where all three peaks occur (Figure~\ref{fig:thstress}). With a sliding support (axial restraint only, four points against rigid-body motion) the peaks fall to \V{THSL-conductor-max}, \V{THSL-core-max} and \V{THSL-flange-max}~MPa. The thermal stress values are therefore sensitive to the mounting stiffness and to the assumed stress-free temperature, and they are indicative only.

\begin{figure}[!htb]\centering
\includegraphics[width=0.56\textwidth]{fig6_thermal_stress_flange.png}
\caption{Thermal von Mises stress near the flange with fixed and sliding flange face (same colour scale).}\label{fig:thstress}
\end{figure}

\section{Mesh convergence}
\begin{table}[!htb]\centering\small
\caption{Mesh convergence for S1 and $f_1$. ``VM away'' excludes points within \V{excl-mm}~mm of singular edges.}\label{tab:conv}
\begin{adjustbox}{max width=\textwidth}\input{generated/convergence_table.tex}\end{adjustbox}
\end{table}
From the medium to the fine mesh the top deflection changes by \V{cv-chg-topdeflectionpct}\,\% and $f_1$ by \V{cv-chg-f1pct}\,\%, so both are converged at the medium mesh used for the results. The stress picture is less clean. The peak stresses grow steadily with refinement in every body (epoxy \V{cv-coarse-core-max}, \V{cv-medium-core-max}, \V{cv-fine-core-max}~MPa; flange \V{cv-coarse-flange-max}, \V{cv-medium-flange-max}, \V{cv-fine-flange-max}~MPa), which is the expected behaviour at singular corners: they do not converge and are not design values. The maxima at \V{excl-mm}~mm from the singular edges do not grow, and in the copper they are converged (\V{cv-chg-conductor-away}\,\%). In the epoxy and the flange they still move by \V{cv-chg-core-away}\,\% and \V{cv-chg-flange-away}\,\% (epoxy \V{cv-coarse-core-away}, \V{cv-medium-core-away}, \V{cv-fine-core-away}~MPa), because \V{excl-mm}~mm is only one or two elements in the flange zone and the sampling point sits in a steep gradient. These local values are small in absolute terms but are not mesh-converged; a reliable local stress needs a local mesh refinement around the actual fillet radii, or a linearised or hot-spot stress method.

\begin{figure}[!htb]\centering
\includegraphics[width=0.78\textwidth]{fig7_mesh_convergence.png}
\caption{Mesh convergence of S1 top deflection, S1 von Mises stress and $f_1$.}\label{fig:conv}
\end{figure}

\section{Limitations and next steps}
\begin{itemize}
\item Real geometry: sheds and housing, condenser foils, the oil-side part, the draw lead and the terminal.
\item Mounting: bolted flange on a flexible tank cover, contact and preload instead of a fixed, bonded face.
\item Seismic: response spectrum analysis to IEEE~693 with tank amplification, damping and modal combination, instead of a static-equivalent acceleration.
\item Short-circuit electromagnetic forces on the conductor and terminal, and the adiabatic short-circuit temperature rise.
\item Thermal and materials: temperature-dependent resistivity and properties, dielectric losses, radiation, heat transfer at the terminal and oil side, cure temperature and residual stress of the resin, allowables.
\item Validation against test data; hand calculations only verify that the equations are solved correctly.
\end{itemize}

\section{References}
{\small
\begin{enumerate}[label={[\arabic*]}]
\item IEC 60137, \emph{Insulated bushings for alternating voltages above 1000 V}.
\item IEEE Std 693, \emph{IEEE Recommended Practice for Seismic Design of Substations}.
\item FEniCSx project, dolfinx documentation, \url{https://docs.fenicsproject.org}.
\item C. Geuzaine and J.-F. Remacle, Gmsh documentation, \url{https://gmsh.info}.
\item SLEPc documentation, \url{https://slepc.upv.es}.
\item W. C. Young and R. G. Budynas, \emph{Roark's Formulas for Stress and Strain}, McGraw-Hill (cantilever deflection, beam frequencies).
\item F. P. Incropera et al., \emph{Fundamentals of Heat and Mass Transfer}, Wiley (radial conduction with heat generation, thermal resistance of cylinders).
\end{enumerate}}

\end{document}

## 6. Smoke test

In [ ]:
import sys
!{sys.executable} src/smoke_test.py

## 7. Run the full study
The fine convergence mesh (h = 13.5 mm, about 1 million degrees of freedom) needs about 13 GB of RAM with the
direct solver. On a machine with less memory the fine size is set to 16 mm; this change is written into
`config.yaml` and therefore recorded in `results.json`.

In [ ]:
import os, re
ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
print(f"RAM: {ram_gb:.1f} GB")
if ram_gb < 20:
    txt = open("config.yaml").read()
    txt = re.sub(r"convergence_h: \[.*?\]", "convergence_h: [0.030, 0.020, 0.016]", txt)
    open("config.yaml", "w").write(txt)
    print("fine convergence mesh set to h = 16 mm for this machine")

In [ ]:
import subprocess, sys
ret = subprocess.run([sys.executable, "src/run_all.py"]).returncode
assert ret == 0, f"run_all.py failed with exit code {ret}"

## 8. Results

In [ ]:
import json
r = json.load(open("results/results.json"))
print(f"S1 top deflection: {r['static']['S1_cantilever']['top_deflection_x_m'] * 1e3:.3f} mm")
print("Natural frequencies (Hz):", [round(m['frequency_Hz'], 2) for m in r['modal']['modes']])
print(f"Hot spot: {r['thermal']['T_max_C']:.1f} C")
print()
print(f"{'Check':95s} {'Analytical':>11s} {'FEA':>10s} {'Diff %':>8s}")
for row in r["verification"]:
    print(f"{row['check'] + ' (' + row['unit'] + ')':95s} {row['analytical']:11.4f} {row['fea']:10.4f} {row['difference_pct']:+8.2f}")

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("figures/*.png")):
    print(f)
    display(Image(filename=f, width=750))

In [ ]:
from IPython.display import Markdown
display(Markdown(open("docs/CV_ENTRY.md").read()))

## 9. Optional: compile the report to PDF
Installs a LaTeX distribution with apt (about 5 minutes on Colab).

In [ ]:
!apt-get -qq update && apt-get -qq install -y --no-install-recommends texlive-latex-recommended texlive-latex-extra texlive-science texlive-fonts-recommended > /dev/null
%cd report
!pdflatex -interaction=nonstopmode report.tex > /dev/null; pdflatex -interaction=nonstopmode report.tex | tail -2
%cd ..

## 10. Download everything as a zip

In [ ]:
!zip -qr bushing-fea-results.zip results/results.json figures report docs README.md config.yaml src
try:
    from google.colab import files
    files.download("bushing-fea-results.zip")
except ImportError:
    print("written bushing-fea-results.zip")